# NeuroScope AI — Acquire the best qualifying disease datasets

**Acquisition preparation, not training.** Run actual transfers only in Google Colab with private Google Drive storage. Never run these transfers in Codex Cloud. All disease switches default to **False**.

This notebook reuses the runtime, dependency, compatibility and Drive setup code from `Brain_MRI_Detection_Colab.ipynb` on `feature/detection-setup` (base commit `9762757efddcc8a749400c050e52d2c169c9d0ef`). The original notebook is unchanged. Helpers and the pinned catalog are embedded so this notebook works without cloning the repository or downloading executable code.

**Three acquisition choices are prepared; two remain unresolved.** Do not interpret the filename as five verified clinical cohorts. Review `detection/research/DATASET_SELECTION.md` for the comparisons, current terms and limitations. The catalog distinguishes source verification from archive download and clinical validation.

Existing **578 IXI-T2 files** at `/content/drive/MyDrive/BrainMRI/datasets/IXI/Healthy/IXI-T2/` are left entirely alone: no download, scan, manifest, rename, move or modification of IXI is performed.

Keep all notebook outputs private. Clear outputs before committing. Never upload MRI files, patient manifests, clinical tables, credentials or weights to GitHub.


## Selection, checked 2026-10-21

| Disease | Prepared selection | Independent patients / case groups | Download | Permission / limitation |
|---|---|---|---|---|
| Glioma | MSD Task01 BrainTumour, MONAI-authorized S3 mirror | 750 case volumes; 484 annotated, 266 without public masks. Independent clinical identities not independently established. | 7,608,266,240 bytes (7.61 GB) | CC BY-SA 4.0; overlaps BraTS 2016/17 |
| Meningioma | OpenNeuro ds007345 1.0.0, baseline structural/mask subset | 37 independent patients, 179 files | 4,407,701,072 bytes (4.41 GB) | CC0; single center and olfactory-groove subtype only |
| Brain metastasis | **Disabled: no verified qualifying direct transfer selected** | Unknown for a selected cohort | Unknown | UCSF-BMSR current endpoint/terms unresolved; UCSD alternative is Aspera |
| Alzheimer's disease | **Disabled: no verified qualifying clinical whole-brain cohort selected** | Unknown | Unknown | OASIS current terms/access unresolved; ADNI/AIBL gated; ex-vivo OpenNeuro is unsuitable |
| Post-stroke lesions | ISLES 2022 public training release | 250 expert-annotated cases/patients from three centers | 1,692,717,470 bytes (1.69 GB) | CC BY 4.0; acute/subacute ischemic lesions, not chronic ATLAS lesions |

No assertion is made that a qualifying dataset cannot exist for either unresolved disease. Their switches fail explicitly rather than downloading an unverified substitute. A dataset's license is separate from the license of a paper or code repository. MD5 checks come from the publisher over HTTPS; a new local SHA-256 records the acquired bytes but is not a substitute for a publisher checksum.


## 1. Configure setup

Keep the default GPU requirement in Colab. For a lightweight CPU check, set
`REQUIRE_GPU = False`, `INSTALL_COLAB_DEPENDENCIES = False`, and `MOUNT_GOOGLE_DRIVE = False`.
Edit `PROJECT_ROOT` to your private project folder. Optional overrides can point to an existing dataset
and a future detection checkpoint. Missing data or weights do not prevent setup checks.


In [ ]:
from pathlib import Path

REQUIRE_GPU = True  # @param {type:"boolean"}
INSTALL_COLAB_DEPENDENCIES = True  # @param {type:"boolean"}
MOUNT_GOOGLE_DRIVE = True  # @param {type:"boolean"}
CREATE_OUTPUT_DIRS = False  # @param {type:"boolean"}

DRIVE_MOUNT_POINT = "/content/drive"  # @param {type:"string"}
PROJECT_ROOT = "/content/drive/MyDrive/BrainMRI"  # @param {type:"string"}
DATASET_ROOT_OVERRIDE = ""  # @param {type:"string"}
CHECKPOINT_PATH_OVERRIDE = ""  # @param {type:"string"}

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Google Colab runtime:", IN_COLAB)


## 2. Check PyTorch, CUDA, and GPU

Use Colab's preinstalled CUDA-enabled PyTorch. This cell checks a tiny tensor operation on the selected
device; it does not create a model. If CUDA is unavailable, select a T4 runtime and reconnect.


In [ ]:
import sys
import torch

def check_torch_runtime(torch_module, require_gpu=True):
    cuda_available = torch_module.cuda.is_available()
    if require_gpu and not cuda_available:
        raise RuntimeError(
            "CUDA is unavailable. In Colab select Runtime > Change runtime type > "
            "T4 GPU and reconnect. Use REQUIRE_GPU=False only for CPU setup checks."
        )

    device = torch_module.device("cuda:0" if cuda_available else "cpu")
    with torch_module.no_grad():
        sample = torch_module.arange(4, dtype=torch_module.float32, device=device)
        if (sample * sample).sum().item() != 14.0:
            raise RuntimeError("PyTorch device arithmetic check failed.")
    if cuda_available:
        torch_module.cuda.synchronize()

    info = {
        "python": sys.version.split()[0],
        "pytorch": str(torch_module.__version__),
        "pytorch_cuda_build": torch_module.version.cuda,
        "cuda_available": cuda_available,
        "device": str(device),
        "gpu_name": torch_module.cuda.get_device_name(0) if cuda_available else None,
        "gpu_memory_gib": (
            round(torch_module.cuda.get_device_properties(0).total_memory / 2**30, 2)
            if cuda_available else None
        ),
        "device_tensor_check": "PASS",
    }
    return info

runtime_info = check_torch_runtime(torch, require_gpu=REQUIRE_GPU)
for key, value in runtime_info.items():
    print(f"{key}: {value}")
if runtime_info["gpu_name"] and "T4" not in runtime_info["gpu_name"]:
    print("This runtime has a different GPU; the project default is Tesla T4.")


## 3. Prepare MONAI and NiBabel

Only in Colab, install **MONAI 1.5.1** and **NiBabel 5.3.2**. PyTorch, CUDA, and NumPy remain supplied
by Colab; `--no-deps` prevents this step from replacing them. The next cell checks the installed
package requirements and runs small in-memory compatibility checks. If a requirement fails, stop
and use a compatible Colab runtime. If either library was already imported at a different version,
restart the session and rerun the notebook.


In [ ]:
import importlib.metadata as metadata
import subprocess

COLAB_PACKAGES = {"monai": "1.5.1", "nibabel": "5.3.2"}

if IN_COLAB and INSTALL_COLAB_DEPENDENCIES:
    requested = []
    for package, expected_version in COLAB_PACKAGES.items():
        try:
            current_version = metadata.version(package)
        except metadata.PackageNotFoundError:
            current_version = None
        if current_version != expected_version:
            requested.append(f"{package}=={expected_version}")
    if requested:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install",
            "--disable-pip-version-check", "--no-input", "--no-deps", *requested,
        ])
    else:
        print("Pinned Colab packages are already installed.")
else:
    print("Dependency installation skipped; existing packages will be checked.")


In [ ]:
import numpy as np
import monai
import nibabel as nib
from packaging.requirements import Requirement
from monai.transforms import NormalizeIntensity

def check_declared_requirements(package_names):
    problems = []
    for package in package_names:
        for requirement_text in metadata.requires(package) or []:
            requirement = Requirement(requirement_text)
            if requirement.marker and not requirement.marker.evaluate({"extra": ""}):
                continue
            try:
                installed_version = metadata.version(requirement.name)
            except metadata.PackageNotFoundError:
                problems.append(f"{package}: missing {requirement.name}")
                continue
            if requirement.specifier and not requirement.specifier.contains(installed_version):
                problems.append(
                    f"{package}: {requirement.name} {installed_version} "
                    f"does not satisfy {requirement.specifier}"
                )
    if problems:
        raise RuntimeError("Package compatibility check failed:\n" + "\n".join(problems))

check_declared_requirements(["torch", "monai", "nibabel"])
for package, module in [("monai", monai), ("nibabel", nib)]:
    if module.__version__ != metadata.version(package):
        raise RuntimeError(f"{package} was updated after import; restart the Colab session.")
    if IN_COLAB and INSTALL_COLAB_DEPENDENCIES:
        if module.__version__ != COLAB_PACKAGES[package]:
            raise RuntimeError(f"Unexpected {package} version; restart and rerun setup.")

# Artificial arrays only; no MRI files or weights are read.
synthetic = np.arange(64, dtype=np.float32).reshape(4, 4, 4)
normalized = NormalizeIntensity()(synthetic)
normalized_tensor = torch.as_tensor(normalized)
assert tuple(normalized_tensor.shape) == (4, 4, 4)
assert torch.isfinite(normalized_tensor).all().item()
assert abs(normalized_tensor.mean().item()) < 1e-5

nifti = nib.Nifti1Image(synthetic, np.eye(4))
assert nifti.shape == (4, 4, 4)
np.testing.assert_array_equal(np.asarray(nifti.dataobj), synthetic)

compatibility_info = {
    "numpy": np.__version__,
    "monai": monai.__version__,
    "nibabel": nib.__version__,
    "declared_requirements": "PASS",
    "monai_transform": "PASS",
    "nifti_in_memory": "PASS",
}
for key, value in compatibility_info.items():
    print(f"{key}: {value}")


## 4. Optionally mount Google Drive

Mounting requests access through Colab's own authorization prompt. Do not put credentials in this
notebook. To work without Drive, disable mounting and change `PROJECT_ROOT` to a local Colab path
such as `/content/brain-mri-detection`. Local Colab files are temporary.


In [ ]:
DRIVE_MOUNTED = False
if MOUNT_GOOGLE_DRIVE:
    if not IN_COLAB:
        print("Drive mounting skipped: this is not a Google Colab runtime.")
    else:
        from google.colab import drive
        drive.mount(DRIVE_MOUNT_POINT, force_remount=False)
        DRIVE_MOUNTED = (Path(DRIVE_MOUNT_POINT) / "MyDrive").is_dir()
        if not DRIVE_MOUNTED:
            raise RuntimeError("Google Drive MyDrive is unavailable after mounting.")
else:
    print("Google Drive mounting disabled.")


## Select diseases and storage limits

Set only the diseases you want to download. For each enabled disease, `Drive/datasets/DiseaseAcquisition/<dataset_id>/` stores downloads, data, source/license metadata, checksum records and manifests. No existing file with a different checksum is overwritten. All original ZIP/TAR files are retained for reproducibility and resumption.

**Drive quota is not reliably reflected by `shutil.disk_usage` on the mounted Drive.** Enter the available GB shown in your Google Drive storage page in `DRIVE_AVAILABLE_GIB` (convert to GiB if necessary). A zero value blocks transfers. The notebook also checks filesystem free space. The preflight reserves the selected download bytes plus the configured upper extraction budget for each archive and 2 GiB headroom. Lower budgets only when you have established actual needs. These are bounds, not published extracted sizes.

Only structural baseline meningioma images and available tumor/edema masks are selected, not diffusion/fMRI or repeat visits. The pinned hashes refuse silently changed OpenNeuro objects. Resumption uses HTTP Range + If-Range where a strong ETag or Last-Modified exists; otherwise partial files require inspection. Existing completed files are rehashed before reuse.


In [ ]:
DOWNLOAD_GLIOMA = False  # @param {type:"boolean"}
DOWNLOAD_MENINGIOMA = False  # @param {type:"boolean"}
DOWNLOAD_BRAIN_METASTASIS = False  # @param {type:"boolean"}
DOWNLOAD_ALZHEIMERS = False  # @param {type:"boolean"}
DOWNLOAD_POST_STROKE = False  # @param {type:"boolean"}
DRIVE_AVAILABLE_GIB = 0.0  # @param {type:"number"}
MAX_DOWNLOAD_GIB_PER_DISEASE = 20.0  # @param {type:"number"}
MAX_UNPACKED_GIB_PER_ARCHIVE = 30.0  # @param {type:"number"}
ACQUISITION_ROOT = str(Path(PROJECT_ROOT) / "datasets" / "DiseaseAcquisition")
SWITCHES = {
    "glioma": DOWNLOAD_GLIOMA, "meningioma": DOWNLOAD_MENINGIOMA,
    "brain_metastasis": DOWNLOAD_BRAIN_METASTASIS, "alzheimers": DOWNLOAD_ALZHEIMERS,
    "post_stroke": DOWNLOAD_POST_STROKE,
}


In [ ]:
import json
CATALOG = json.loads('{"glioma": {"id": "msd_task01", "disease": "Glioma", "eligible": true, "selection": "Practical authorized direct-download choice; not a claim of largest available cohort", "source": "https://medicaldecathlon.com/", "authorized_mirror_evidence": "https://raw.githubusercontent.com/Project-MONAI/MONAI/1.5.1/monai/apps/datasets.py", "license": "CC-BY-SA-4.0", "license_url": "https://creativecommons.org/licenses/by-sa/4.0/", "citation": "Antonelli et al. The Medical Segmentation Decathlon. Nature Communications 13, 4128 (2022). doi:10.1038/s41467-022-30695-9; cite BraTS source publications too.", "independent_patients": null, "expected_groups": 750, "labeled_cases": 484, "unlabeled_cases": 266, "count_note": "750 4D case volumes, not 3,000 patients. Unique clinical patient identity not independently established. BraTS 2016/17 overlap; do not pool as independent cohorts.", "download_probe": "200 HEAD, Content-Length 7608266240, Accept-Ranges bytes; official MONAI 1.5.1 MD5 verified; payload not fetched.", "assets": [{"path": "Task01_BrainTumour.tar", "url": "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task01_BrainTumour.tar", "bytes": 7608266240, "checksum": "md5:240a19d752f0d9e9101544901065d872", "archive": true}], "limitations": ["7.61 GB archive (7.09 GiB), confirmed by HEAD; extraction needs additional storage. The previous notebook 2 GB estimate is incorrect.", "Training masks only for 484 cases; not all 750.", "Already represented in the segmentation project: reuse any existing verified archive in this acquisition destination; do not duplicate under BraTS names.", "License must also be confirmed from extracted dataset.json; no clinical approval is implied."], "download_bytes": 7608266240, "expected_min_masks": 484, "license_evidence": "https://medicaldecathlon.com/"}, "meningioma": {"id": "openneuro_ds007345", "disease": "Meningioma", "eligible": true, "selection": "Verified ungated structural-MRI alternative to larger controlled-access releases", "source": "https://openneuro.org/datasets/ds007345/versions/1.0.0", "metadata_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/dataset_description.json", "license": "CC0-1.0", "license_url": "https://creativecommons.org/publicdomain/zero/1.0/", "citation": "Filimonova et al. Multimodal Brain MRI and Clinical Data in Olfactory Groove Meningioma: A Prospective Data Report. doi:10.18112/openneuro.ds007345.v1.0.0", "independent_patients": 37, "expected_groups": 37, "release": "1.0.0", "metadata_commit": "fd052d49bfa51677fca817efdb68f237eeda9774", "download_bytes": 4407701072, "download_probe": "All 179 selected S3 assets returned HEAD 200 with matching byte sizes and Range support. Every publisher SHA256E pointer was verified. No MRI payload fetched in Cloud.", "assets": [{"path": "derivatives/sub-olf01/segmentation/sub-olf01_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf01/segmentation/sub-olf01_tumor_mask.nii.gz", "bytes": 105697, "checksum": "sha256:7b7715489e4e2e14f835ace3f1ab17490a08c5d9b05b78b901c376bfefed6910", "patient_id": "sub-olf01", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf01/segmentation/sub-olf01_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf02/segmentation/sub-olf02_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf02/segmentation/sub-olf02_edema_mask.nii.gz", "bytes": 295013, "checksum": "sha256:eb9a4729f85002f99c007357a7b6a6c55d4742ade059ad68773cd34a91bf7207", "patient_id": "sub-olf02", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf02/segmentation/sub-olf02_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf02/segmentation/sub-olf02_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf02/segmentation/sub-olf02_tumor_mask.nii.gz", "bytes": 111809, "checksum": "sha256:4450f6e942b126d7a1da2dd519c9f2e0c02b643db0245de6861aab281d994125", "patient_id": "sub-olf02", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf02/segmentation/sub-olf02_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf03/segmentation/sub-olf03_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf03/segmentation/sub-olf03_edema_mask.nii.gz", "bytes": 201309, "checksum": "sha256:a1d72d7d547d6fe718fb7602c657e18bcd2db56e19d29760071a847ee30c119e", "patient_id": "sub-olf03", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf03/segmentation/sub-olf03_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf03/segmentation/sub-olf03_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf03/segmentation/sub-olf03_tumor_mask.nii.gz", "bytes": 107366, "checksum": "sha256:98737dc16e31186d11f5deb25bdef332cca78f8e5768a862de9277c01c2812ce", "patient_id": "sub-olf03", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf03/segmentation/sub-olf03_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf04/segmentation/sub-olf04_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf04/segmentation/sub-olf04_edema_mask.nii.gz", "bytes": 236203, "checksum": "sha256:c8ee8c006fe0e3cb8373a3ad00cee6eaff77933cf2130e6691b2e0930be0dd08", "patient_id": "sub-olf04", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf04/segmentation/sub-olf04_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf04/segmentation/sub-olf04_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf04/segmentation/sub-olf04_tumor_mask.nii.gz", "bytes": 112985, "checksum": "sha256:97f52f718a777355733928ca04304e527fa338b8b5744ed413eb1f2162aedbf7", "patient_id": "sub-olf04", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf04/segmentation/sub-olf04_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf05/segmentation/sub-olf05_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf05/segmentation/sub-olf05_edema_mask.nii.gz", "bytes": 218413, "checksum": "sha256:62cf87511e4db8c7f91527c165c868db9586bbd08b50678e0993d6ae59ac6c20", "patient_id": "sub-olf05", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf05/segmentation/sub-olf05_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf05/segmentation/sub-olf05_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf05/segmentation/sub-olf05_tumor_mask.nii.gz", "bytes": 105549, "checksum": "sha256:b2e7defff9617c79c253b474edd6f97555101d638090b6f8a424afdac25d7321", "patient_id": "sub-olf05", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf05/segmentation/sub-olf05_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf06/segmentation/sub-olf06_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf06/segmentation/sub-olf06_edema_mask.nii.gz", "bytes": 260088, "checksum": "sha256:83f76e9dd2adb8aa0f25ea64e650194a8d60bbf1dbf0462dd2f594d689ed12d5", "patient_id": "sub-olf06", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf06/segmentation/sub-olf06_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf06/segmentation/sub-olf06_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf06/segmentation/sub-olf06_tumor_mask.nii.gz", "bytes": 118913, "checksum": "sha256:c22ab68f8ba0f454ca7e34f00803584812a0c07769593767f9f3c75fe0ab14aa", "patient_id": "sub-olf06", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf06/segmentation/sub-olf06_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf07/segmentation/sub-olf07_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf07/segmentation/sub-olf07_edema_mask.nii.gz", "bytes": 300002, "checksum": "sha256:0ecd230d74c387ac4ada031fab432b523c76fcb45adae5bd018eb3be0c14a6f0", "patient_id": "sub-olf07", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf07/segmentation/sub-olf07_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf07/segmentation/sub-olf07_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf07/segmentation/sub-olf07_tumor_mask.nii.gz", "bytes": 105704, "checksum": "sha256:26f07920afd03291c867a47249802517a291014f39e22e87c2a8b596c2b0e7d8", "patient_id": "sub-olf07", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf07/segmentation/sub-olf07_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf08/segmentation/sub-olf08_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf08/segmentation/sub-olf08_edema_mask.nii.gz", "bytes": 233012, "checksum": "sha256:efe49c193759f2c36154e966a009cfa684428c6d5c2bc449e1a6be1f380e47ce", "patient_id": "sub-olf08", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf08/segmentation/sub-olf08_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf08/segmentation/sub-olf08_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf08/segmentation/sub-olf08_tumor_mask.nii.gz", "bytes": 124258, "checksum": "sha256:0ad8052df8d1601a88a3f8c446a0cd692ceed5ce4124071fa4b91707d8755764", "patient_id": "sub-olf08", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf08/segmentation/sub-olf08_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf09/segmentation/sub-olf09_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf09/segmentation/sub-olf09_tumor_mask.nii.gz", "bytes": 115778, "checksum": "sha256:315387ceca5786d748c5deb7a8186b4a4f75208b06f432aa5df2e6e8ea5c5a54", "patient_id": "sub-olf09", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf09/segmentation/sub-olf09_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf10/segmentation/sub-olf10_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf10/segmentation/sub-olf10_edema_mask.nii.gz", "bytes": 257317, "checksum": "sha256:018cfff517b27c2db446a76993fb8fd11c74046d31788bf1a40b8b8899f86f52", "patient_id": "sub-olf10", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf10/segmentation/sub-olf10_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf10/segmentation/sub-olf10_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf10/segmentation/sub-olf10_tumor_mask.nii.gz", "bytes": 107075, "checksum": "sha256:767dea0007283121900b34e6e744c9a5461a81c8a5997d21ed78f4fff12ecf80", "patient_id": "sub-olf10", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf10/segmentation/sub-olf10_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf11/segmentation/sub-olf11_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf11/segmentation/sub-olf11_edema_mask.nii.gz", "bytes": 191178, "checksum": "sha256:73698328ce68ea01dfc799853be2b7bbf595ef4d8cb2c2e80d4b04589b06812c", "patient_id": "sub-olf11", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf11/segmentation/sub-olf11_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf11/segmentation/sub-olf11_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf11/segmentation/sub-olf11_tumor_mask.nii.gz", "bytes": 109527, "checksum": "sha256:ee43795fafea7ee445e771c8d3b97a941d7541d8d6845760b83891a062049b77", "patient_id": "sub-olf11", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf11/segmentation/sub-olf11_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf12/segmentation/sub-olf12_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf12/segmentation/sub-olf12_edema_mask.nii.gz", "bytes": 265135, "checksum": "sha256:f3f0ddb62b10c42855b121cc76af959422204b56b5f5039f18311445710ce124", "patient_id": "sub-olf12", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf12/segmentation/sub-olf12_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf12/segmentation/sub-olf12_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf12/segmentation/sub-olf12_tumor_mask.nii.gz", "bytes": 109936, "checksum": "sha256:b32a0709844fdecf55f83b7c30943792c9695f98d9226143377ce113a7fef031", "patient_id": "sub-olf12", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf12/segmentation/sub-olf12_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf13/segmentation/sub-olf13_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf13/segmentation/sub-olf13_tumor_mask.nii.gz", "bytes": 102825, "checksum": "sha256:2defc7a16ddd4e5db6bf36b099b79ffd993d2be74a8ef64b001876568de1f041", "patient_id": "sub-olf13", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf13/segmentation/sub-olf13_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf14/segmentation/sub-olf14_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf14/segmentation/sub-olf14_edema_mask.nii.gz", "bytes": 202848, "checksum": "sha256:11a76022776a5f7341f82439446ff5ac2438c26149b8cf9b1f8a066788003ff7", "patient_id": "sub-olf14", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf14/segmentation/sub-olf14_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf14/segmentation/sub-olf14_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf14/segmentation/sub-olf14_tumor_mask.nii.gz", "bytes": 102052, "checksum": "sha256:ddf0bdd50204cab98af56141581a901e9427d51ab8c42d03ebbf48336f063f45", "patient_id": "sub-olf14", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf14/segmentation/sub-olf14_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf15/segmentation/sub-olf15_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf15/segmentation/sub-olf15_tumor_mask.nii.gz", "bytes": 122204, "checksum": "sha256:b6c5a4582aeb3f800c29098361ba5ad16e10ce63ea796876e5018ea8fe99af89", "patient_id": "sub-olf15", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf15/segmentation/sub-olf15_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf16/segmentation/sub-olf16_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf16/segmentation/sub-olf16_edema_mask.nii.gz", "bytes": 195727, "checksum": "sha256:16374699cd5b6f916528ccd002a20abfa107be4ab700572b9a85139bbef61cf1", "patient_id": "sub-olf16", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf16/segmentation/sub-olf16_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf16/segmentation/sub-olf16_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf16/segmentation/sub-olf16_tumor_mask.nii.gz", "bytes": 105007, "checksum": "sha256:2eb55f9e707e60f26f24143bdacdf10169a379410ac891b1334517aef23b8c64", "patient_id": "sub-olf16", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf16/segmentation/sub-olf16_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf17/segmentation/sub-olf17_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf17/segmentation/sub-olf17_edema_mask.nii.gz", "bytes": 283446, "checksum": "sha256:0078b418e62a8489540b86746d1fed722b4a19040c87fee84196ed5bfac881a3", "patient_id": "sub-olf17", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf17/segmentation/sub-olf17_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf17/segmentation/sub-olf17_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf17/segmentation/sub-olf17_tumor_mask.nii.gz", "bytes": 110416, "checksum": "sha256:1ef8ca06990f07806b384294438e17e91948394153be2a70588314c0d8619fe1", "patient_id": "sub-olf17", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf17/segmentation/sub-olf17_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf18/segmentation/sub-olf18_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf18/segmentation/sub-olf18_edema_mask.nii.gz", "bytes": 230844, "checksum": "sha256:8e96a17d42f1835e168853bdec1663c713a089f3d756370362db29a981cfbb90", "patient_id": "sub-olf18", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf18/segmentation/sub-olf18_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf18/segmentation/sub-olf18_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf18/segmentation/sub-olf18_tumor_mask.nii.gz", "bytes": 119233, "checksum": "sha256:75167b60c252bd971ba32c96afb57809986e7dbecc11ee7c0d98f346263e8c31", "patient_id": "sub-olf18", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf18/segmentation/sub-olf18_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf19/segmentation/sub-olf19_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf19/segmentation/sub-olf19_edema_mask.nii.gz", "bytes": 29542, "checksum": "sha256:bf3cce67d7a283eb08d710cb1e95b0315d093ac9649679d5e6f794e222fe635b", "patient_id": "sub-olf19", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf19/segmentation/sub-olf19_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf19/segmentation/sub-olf19_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf19/segmentation/sub-olf19_tumor_mask.nii.gz", "bytes": 143211, "checksum": "sha256:061b3d683c0068b649933868386c1bbfa76bc70f5e347f80b224a81cced6d04d", "patient_id": "sub-olf19", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf19/segmentation/sub-olf19_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf20/segmentation/sub-olf20_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf20/segmentation/sub-olf20_edema_mask.nii.gz", "bytes": 282350, "checksum": "sha256:03203f50ec0d57d1178212a47db79b4fd0fd0e71c2d36189a27b01e5583e6807", "patient_id": "sub-olf20", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf20/segmentation/sub-olf20_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf20/segmentation/sub-olf20_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf20/segmentation/sub-olf20_tumor_mask.nii.gz", "bytes": 114631, "checksum": "sha256:5bc21c27aaa780f9570d0a11ed9e307452200a639d3083d0ff33375e6d4653d3", "patient_id": "sub-olf20", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf20/segmentation/sub-olf20_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf21/segmentation/sub-olf21_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf21/segmentation/sub-olf21_edema_mask.nii.gz", "bytes": 352158, "checksum": "sha256:91ac73b95b3f76295b5318a76fd6a19eee316b7417b7d17f01e15190a0cc3f2c", "patient_id": "sub-olf21", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf21/segmentation/sub-olf21_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf21/segmentation/sub-olf21_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf21/segmentation/sub-olf21_tumor_mask.nii.gz", "bytes": 100373, "checksum": "sha256:a60aa71b3a746f7d8dd53b1df18c1c426da6dc9c0edb609a60e8532f9a79c4a3", "patient_id": "sub-olf21", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf21/segmentation/sub-olf21_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf22/segmentation/sub-olf22_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf22/segmentation/sub-olf22_edema_mask.nii.gz", "bytes": 307415, "checksum": "sha256:f95817b52eac943c6ccacd869ea5116bd1984795f53703605481c93ee19bdca5", "patient_id": "sub-olf22", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf22/segmentation/sub-olf22_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf22/segmentation/sub-olf22_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf22/segmentation/sub-olf22_tumor_mask.nii.gz", "bytes": 110879, "checksum": "sha256:3291adc0176bd5aa58496bbcd6ce0c664b7f3669ed854f25430ce32abdb90de6", "patient_id": "sub-olf22", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf22/segmentation/sub-olf22_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf23/segmentation/sub-olf23_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf23/segmentation/sub-olf23_edema_mask.nii.gz", "bytes": 222059, "checksum": "sha256:03f075ec990f0eed566ee93553e9951fdf0142bddda3f53ad946ab5d87a55b14", "patient_id": "sub-olf23", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf23/segmentation/sub-olf23_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf23/segmentation/sub-olf23_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf23/segmentation/sub-olf23_tumor_mask.nii.gz", "bytes": 105382, "checksum": "sha256:b6ca733cf84cec84e52dca35efc5e449b69e3451e45f2e2ac36a373dad5e69d6", "patient_id": "sub-olf23", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf23/segmentation/sub-olf23_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf24/segmentation/sub-olf24_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf24/segmentation/sub-olf24_edema_mask.nii.gz", "bytes": 191627, "checksum": "sha256:361f73154861d398698aaddbbb757aeb0d14ae3cf48af5244e50a439139f6258", "patient_id": "sub-olf24", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf24/segmentation/sub-olf24_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf24/segmentation/sub-olf24_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf24/segmentation/sub-olf24_tumor_mask.nii.gz", "bytes": 125947, "checksum": "sha256:75560e39e838ad7e4a9106b00b686b32922717b4ce47cc5e3c3a5a42171e1e2c", "patient_id": "sub-olf24", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf24/segmentation/sub-olf24_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf25/segmentation/sub-olf25_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf25/segmentation/sub-olf25_edema_mask.nii.gz", "bytes": 244715, "checksum": "sha256:29672f2e9ad6c573794be6ae474060f3e6923648b477e117faeff2a61a4dc464", "patient_id": "sub-olf25", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf25/segmentation/sub-olf25_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf25/segmentation/sub-olf25_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf25/segmentation/sub-olf25_tumor_mask.nii.gz", "bytes": 107978, "checksum": "sha256:6fdc221d526ff0277ffe6ff4abee3fb4caef37b4ae1ed02eb5d826d6860f74ef", "patient_id": "sub-olf25", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf25/segmentation/sub-olf25_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf26/segmentation/sub-olf26_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf26/segmentation/sub-olf26_edema_mask.nii.gz", "bytes": 191910, "checksum": "sha256:6cc3e5343132f0d3aa4a83cc31516ef437d241fe9367a436557e1c733d680a97", "patient_id": "sub-olf26", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf26/segmentation/sub-olf26_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf26/segmentation/sub-olf26_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf26/segmentation/sub-olf26_tumor_mask.nii.gz", "bytes": 103688, "checksum": "sha256:e58a1d41a7f2889ff25170ada6d35c5b61b2c61733b8d325a249a875ee27e537", "patient_id": "sub-olf26", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf26/segmentation/sub-olf26_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf27/segmentation/sub-olf27_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf27/segmentation/sub-olf27_edema_mask.nii.gz", "bytes": 272120, "checksum": "sha256:cee39ac1e0539da511e8f950f1c466cd6af295a3010aa464fefe8cbfd3848f79", "patient_id": "sub-olf27", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf27/segmentation/sub-olf27_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf27/segmentation/sub-olf27_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf27/segmentation/sub-olf27_tumor_mask.nii.gz", "bytes": 111513, "checksum": "sha256:67845a9c304d5bb94e10f50963ed3665232865c6faab1897870b735e477d5825", "patient_id": "sub-olf27", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf27/segmentation/sub-olf27_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf28/segmentation/sub-olf28_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf28/segmentation/sub-olf28_edema_mask.nii.gz", "bytes": 258967, "checksum": "sha256:2e7eba8886a3d6e8dac38c630e8547ce3c007b5d37bbda95ae565c32fc898baf", "patient_id": "sub-olf28", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf28/segmentation/sub-olf28_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf28/segmentation/sub-olf28_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf28/segmentation/sub-olf28_tumor_mask.nii.gz", "bytes": 113882, "checksum": "sha256:ebbb65f9eeb998dbf41774b114eda58d4aefd3baf4e61b2da641654aaeb99989", "patient_id": "sub-olf28", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf28/segmentation/sub-olf28_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf29/segmentation/sub-olf29_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf29/segmentation/sub-olf29_edema_mask.nii.gz", "bytes": 219764, "checksum": "sha256:be3655ecf708cfc0a02de99ecf6a23e18ef946e0d2571aaaff3823bad7916bf0", "patient_id": "sub-olf29", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf29/segmentation/sub-olf29_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf29/segmentation/sub-olf29_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf29/segmentation/sub-olf29_tumor_mask.nii.gz", "bytes": 104170, "checksum": "sha256:baa55747690494314e66b4c17524c4e2f3422d4b52f28d582d4fad1cfc6ceadd", "patient_id": "sub-olf29", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf29/segmentation/sub-olf29_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf30/segmentation/sub-olf30_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf30/segmentation/sub-olf30_edema_mask.nii.gz", "bytes": 224352, "checksum": "sha256:896b602c5e38be45c5be2ae23a51892cbf7f3ff275a87d11df1868af61524bbd", "patient_id": "sub-olf30", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf30/segmentation/sub-olf30_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf30/segmentation/sub-olf30_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf30/segmentation/sub-olf30_tumor_mask.nii.gz", "bytes": 124513, "checksum": "sha256:6bbb9890ae61f7b8257964f9518a35339478297a232a4e988b2638e4cb571405", "patient_id": "sub-olf30", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf30/segmentation/sub-olf30_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf31/segmentation/sub-olf31_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf31/segmentation/sub-olf31_edema_mask.nii.gz", "bytes": 264377, "checksum": "sha256:4f7e0f5233a2d6314be9d3745fd66c7465b3942b09228eeecdb7cebcf1076394", "patient_id": "sub-olf31", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf31/segmentation/sub-olf31_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf31/segmentation/sub-olf31_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf31/segmentation/sub-olf31_tumor_mask.nii.gz", "bytes": 113213, "checksum": "sha256:c85c9d0232d9eeadf51296bf775704aa9b62d01787ca0c833944c7d4c9552fdd", "patient_id": "sub-olf31", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf31/segmentation/sub-olf31_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf32/segmentation/sub-olf32_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf32/segmentation/sub-olf32_edema_mask.nii.gz", "bytes": 252579, "checksum": "sha256:e7785a6f0bc961b657e8a10c065fbf7fc779b1588308d7c60b6208aee95c4360", "patient_id": "sub-olf32", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf32/segmentation/sub-olf32_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf32/segmentation/sub-olf32_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf32/segmentation/sub-olf32_tumor_mask.nii.gz", "bytes": 123734, "checksum": "sha256:c6825844502f7f0f09185a3a52d944d74217f91e33966b04672868905844caff", "patient_id": "sub-olf32", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf32/segmentation/sub-olf32_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf33/segmentation/sub-olf33_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf33/segmentation/sub-olf33_tumor_mask.nii.gz", "bytes": 125197, "checksum": "sha256:0197aec088b91466614703e8c2e5b42ccfbda9d98d2717fcc83e2432305f8299", "patient_id": "sub-olf33", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf33/segmentation/sub-olf33_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf34/segmentation/sub-olf34_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf34/segmentation/sub-olf34_tumor_mask.nii.gz", "bytes": 125131, "checksum": "sha256:116894e0cdffef2804617e7076e5920cf31c2084dd93df64d51a8bf0284b4d5e", "patient_id": "sub-olf34", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf34/segmentation/sub-olf34_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf35/segmentation/sub-olf35_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf35/segmentation/sub-olf35_edema_mask.nii.gz", "bytes": 336594, "checksum": "sha256:f0c5178d754051660ebcb439240ea266efbdcbc3737152d92f092e3874dfaa74", "patient_id": "sub-olf35", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf35/segmentation/sub-olf35_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf35/segmentation/sub-olf35_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf35/segmentation/sub-olf35_tumor_mask.nii.gz", "bytes": 107420, "checksum": "sha256:77d96b72ff1147044159be3a17608f9f12ae32e15eb722afc4554f1c1e4935e6", "patient_id": "sub-olf35", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf35/segmentation/sub-olf35_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf36/segmentation/sub-olf36_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf36/segmentation/sub-olf36_edema_mask.nii.gz", "bytes": 225976, "checksum": "sha256:4fea2086e58e53aaa737c442dbe17b09331bb729cd76660d9a31823c80442c5e", "patient_id": "sub-olf36", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf36/segmentation/sub-olf36_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf36/segmentation/sub-olf36_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf36/segmentation/sub-olf36_tumor_mask.nii.gz", "bytes": 121547, "checksum": "sha256:f6078fcb65c2f9d7b29f5c4c3d5900583e7983b3d4b0d5679b81252b7fc0c74b", "patient_id": "sub-olf36", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf36/segmentation/sub-olf36_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf37/segmentation/sub-olf37_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf37/segmentation/sub-olf37_edema_mask.nii.gz", "bytes": 267171, "checksum": "sha256:9b821b1a9b0c00535d1faebe58c503c649a8e4b5ac1a30d46ba175df0bc47c40", "patient_id": "sub-olf37", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf37/segmentation/sub-olf37_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf37/segmentation/sub-olf37_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf37/segmentation/sub-olf37_tumor_mask.nii.gz", "bytes": 105137, "checksum": "sha256:6d9dae8dedf632e9d6da03c64cd86ae7b08c42eb4c807f26dc26afc24f560e4c", "patient_id": "sub-olf37", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf37/segmentation/sub-olf37_tumor_mask.nii.gz"}, {"path": "sub-olf01/anat/sub-olf01_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf01/anat/sub-olf01_T2w.nii.gz", "bytes": 52046803, "checksum": "sha256:f61430a825045f5d7a4011a16e5982a3ef59745fa169b5ad3b6afafb07b03c0f", "patient_id": "sub-olf01", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf01/anat/sub-olf01_T2w.nii.gz"}, {"path": "sub-olf01/anat/sub-olf01_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf01/anat/sub-olf01_run-01_T1w.nii.gz", "bytes": 26564886, "checksum": "sha256:32fab159e3f8273a58448740fecd6c36e85c83c8a2bf6183982678d4b72cadf7", "patient_id": "sub-olf01", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf01/anat/sub-olf01_run-01_T1w.nii.gz"}, {"path": "sub-olf01/anat/sub-olf01_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf01/anat/sub-olf01_run-02_T1w.nii.gz", "bytes": 29136633, "checksum": "sha256:16136a0c473cc44cdaab681131ea681e56384c99e7e826f30397c697c73e6e3d", "patient_id": "sub-olf01", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf01/anat/sub-olf01_run-02_T1w.nii.gz"}, {"path": "sub-olf02/anat/sub-olf02_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf02/anat/sub-olf02_T2w.nii.gz", "bytes": 54703793, "checksum": "sha256:df0b1beb7922e3862d2ad73f07172e1ce8836978b949e421410d260e0564a268", "patient_id": "sub-olf02", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf02/anat/sub-olf02_T2w.nii.gz"}, {"path": "sub-olf02/anat/sub-olf02_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf02/anat/sub-olf02_run-01_T1w.nii.gz", "bytes": 26493380, "checksum": "sha256:a0b7a01f9aa80712ee141afad2176cc279bc3abf0da1bb777c62f26827197a35", "patient_id": "sub-olf02", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf02/anat/sub-olf02_run-01_T1w.nii.gz"}, {"path": "sub-olf02/anat/sub-olf02_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf02/anat/sub-olf02_run-02_T1w.nii.gz", "bytes": 32703957, "checksum": "sha256:627881f4876ff2a39460dc74396315482fd24a50ad363947fe932754b8b8533a", "patient_id": "sub-olf02", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf02/anat/sub-olf02_run-02_T1w.nii.gz"}, {"path": "sub-olf03/anat/sub-olf03_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf03/anat/sub-olf03_T2w.nii.gz", "bytes": 49290364, "checksum": "sha256:c4a77201c7795a5beca141a1a007c4e92b18b7949961723cf139d21c0d30a204", "patient_id": "sub-olf03", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf03/anat/sub-olf03_T2w.nii.gz"}, {"path": "sub-olf03/anat/sub-olf03_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf03/anat/sub-olf03_run-01_T1w.nii.gz", "bytes": 26434171, "checksum": "sha256:61826fb78334ac14a566809d364967dd06cee43a0321994c9d06ccd57018cad0", "patient_id": "sub-olf03", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf03/anat/sub-olf03_run-01_T1w.nii.gz"}, {"path": "sub-olf03/anat/sub-olf03_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf03/anat/sub-olf03_run-02_T1w.nii.gz", "bytes": 28519099, "checksum": "sha256:9aec6ca7b7f0d183ef42d912a9be40f67702549c953767fbacfe1cd4b4925967", "patient_id": "sub-olf03", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf03/anat/sub-olf03_run-02_T1w.nii.gz"}, {"path": "sub-olf04/anat/sub-olf04_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf04/anat/sub-olf04_T2w.nii.gz", "bytes": 54644197, "checksum": "sha256:17864eb8585dc304d977384f5d5ade448af1d1a53f80eefc649c7aa34c296ba3", "patient_id": "sub-olf04", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf04/anat/sub-olf04_T2w.nii.gz"}, {"path": "sub-olf04/anat/sub-olf04_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf04/anat/sub-olf04_run-01_T1w.nii.gz", "bytes": 26863400, "checksum": "sha256:abef06fa7ebbd3bc8b60dfbd1fb642e537cd1b8128b3412eca7fb50324d73e43", "patient_id": "sub-olf04", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf04/anat/sub-olf04_run-01_T1w.nii.gz"}, {"path": "sub-olf04/anat/sub-olf04_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf04/anat/sub-olf04_run-02_T1w.nii.gz", "bytes": 28148902, "checksum": "sha256:b49eb2329d06fe4ae5f3f01fe6b2a3264a60d6cf9684bd29834b770a14b0c064", "patient_id": "sub-olf04", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf04/anat/sub-olf04_run-02_T1w.nii.gz"}, {"path": "sub-olf05/anat/sub-olf05_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf05/anat/sub-olf05_T2w.nii.gz", "bytes": 57538406, "checksum": "sha256:04e2f15ea51df49aadc680d1c5a2d9c975821963eece378b6966bc60568d1141", "patient_id": "sub-olf05", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf05/anat/sub-olf05_T2w.nii.gz"}, {"path": "sub-olf05/anat/sub-olf05_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf05/anat/sub-olf05_run-01_T1w.nii.gz", "bytes": 27166908, "checksum": "sha256:f888e924bb6fbab164bd2c3e5e95d303f03ce13995a34bda003d326e76382920", "patient_id": "sub-olf05", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf05/anat/sub-olf05_run-01_T1w.nii.gz"}, {"path": "sub-olf05/anat/sub-olf05_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf05/anat/sub-olf05_run-02_T1w.nii.gz", "bytes": 29808174, "checksum": "sha256:5fd2ed6d0966a74d710738bb1612556a5943f304adb3def473a8563a542bab94", "patient_id": "sub-olf05", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf05/anat/sub-olf05_run-02_T1w.nii.gz"}, {"path": "sub-olf06/anat/sub-olf06_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf06/anat/sub-olf06_T2w.nii.gz", "bytes": 60424073, "checksum": "sha256:0e3178016846eb76bf746c8f606a0401222245288386f2dde7d854c11ba11c99", "patient_id": "sub-olf06", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf06/anat/sub-olf06_T2w.nii.gz"}, {"path": "sub-olf06/anat/sub-olf06_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf06/anat/sub-olf06_run-01_T1w.nii.gz", "bytes": 29533500, "checksum": "sha256:21a88dbff98726922a2a379d9bf6e360b37b9960ed2f8b1353d11654601f7391", "patient_id": "sub-olf06", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf06/anat/sub-olf06_run-01_T1w.nii.gz"}, {"path": "sub-olf06/anat/sub-olf06_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf06/anat/sub-olf06_run-02_T1w.nii.gz", "bytes": 34469078, "checksum": "sha256:8b44047473a4d6dada7bd606b18b32e98e771cc31315e8d8b33333506fce083b", "patient_id": "sub-olf06", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf06/anat/sub-olf06_run-02_T1w.nii.gz"}, {"path": "sub-olf07/anat/sub-olf07_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf07/anat/sub-olf07_T2w.nii.gz", "bytes": 53090990, "checksum": "sha256:a3b5d6a31438f3b4a74dd20e7e95dab667f547cdfc078d5b07ade26ce1869f8c", "patient_id": "sub-olf07", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf07/anat/sub-olf07_T2w.nii.gz"}, {"path": "sub-olf07/anat/sub-olf07_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf07/anat/sub-olf07_run-01_T1w.nii.gz", "bytes": 27203938, "checksum": "sha256:51fc16024823441838abf30ff9cd1f3589a0cef8eddd338841253d1712048326", "patient_id": "sub-olf07", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf07/anat/sub-olf07_run-01_T1w.nii.gz"}, {"path": "sub-olf07/anat/sub-olf07_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf07/anat/sub-olf07_run-02_T1w.nii.gz", "bytes": 29164678, "checksum": "sha256:f78c80648cea18793f6a4f1229b31202872dd12ab65722c777c0d7a956ee424d", "patient_id": "sub-olf07", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf07/anat/sub-olf07_run-02_T1w.nii.gz"}, {"path": "sub-olf08/anat/sub-olf08_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf08/anat/sub-olf08_T2w.nii.gz", "bytes": 48387585, "checksum": "sha256:5b120d6ddf649120dcf82d8e5df2f9d0ab732eb44cb11611113dae59fb2c09a8", "patient_id": "sub-olf08", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf08/anat/sub-olf08_T2w.nii.gz"}, {"path": "sub-olf08/anat/sub-olf08_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf08/anat/sub-olf08_run-01_T1w.nii.gz", "bytes": 25366316, "checksum": "sha256:a12ebd36f95c3830dd48ee4532c33e64b2311c3b75bb137191d4bec5fa8e46c7", "patient_id": "sub-olf08", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf08/anat/sub-olf08_run-01_T1w.nii.gz"}, {"path": "sub-olf08/anat/sub-olf08_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf08/anat/sub-olf08_run-02_T1w.nii.gz", "bytes": 26557014, "checksum": "sha256:cb3677deab00eb8145d0f8a83b6f26fe591a271923d272a274f1749c2536c8de", "patient_id": "sub-olf08", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf08/anat/sub-olf08_run-02_T1w.nii.gz"}, {"path": "sub-olf09/anat/sub-olf09_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf09/anat/sub-olf09_T2w.nii.gz", "bytes": 59341057, "checksum": "sha256:431edf40769d87e1b0b2419d7796f074ebdfd365e1f088e28de9dbbd1d48afc0", "patient_id": "sub-olf09", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf09/anat/sub-olf09_T2w.nii.gz"}, {"path": "sub-olf09/anat/sub-olf09_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf09/anat/sub-olf09_run-01_T1w.nii.gz", "bytes": 27728670, "checksum": "sha256:2b1cacdc8a5dc4403a249a3a01add2e7813a85049713e3c529c284dff5d875a0", "patient_id": "sub-olf09", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf09/anat/sub-olf09_run-01_T1w.nii.gz"}, {"path": "sub-olf09/anat/sub-olf09_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf09/anat/sub-olf09_run-02_T1w.nii.gz", "bytes": 32637549, "checksum": "sha256:372627e5f6ef795fdfc4acad6294641b913104b791030501a46a2c82bbde4e82", "patient_id": "sub-olf09", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf09/anat/sub-olf09_run-02_T1w.nii.gz"}, {"path": "sub-olf10/anat/sub-olf10_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf10/anat/sub-olf10_T2w.nii.gz", "bytes": 54746691, "checksum": "sha256:bf4b014e7a94c98480be15fe82c1f6fcdfa201a4a8b3994d0803448ba3be9fe7", "patient_id": "sub-olf10", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf10/anat/sub-olf10_T2w.nii.gz"}, {"path": "sub-olf10/anat/sub-olf10_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf10/anat/sub-olf10_run-01_T1w.nii.gz", "bytes": 25601712, "checksum": "sha256:20c1377df3849aaf6fca8a0e81e67c27ce09a85fb34c513b43cd095eca62c52c", "patient_id": "sub-olf10", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf10/anat/sub-olf10_run-01_T1w.nii.gz"}, {"path": "sub-olf10/anat/sub-olf10_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf10/anat/sub-olf10_run-02_T1w.nii.gz", "bytes": 30659867, "checksum": "sha256:036a84398ada37c2c6523dec3cf01fa16c0c7ace38b751ea26601a80e6b10680", "patient_id": "sub-olf10", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf10/anat/sub-olf10_run-02_T1w.nii.gz"}, {"path": "sub-olf11/anat/sub-olf11_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf11/anat/sub-olf11_T2w.nii.gz", "bytes": 63288734, "checksum": "sha256:523482d4d4a8a276cce0871464d58a5cf5649ae11abaf8d8fdc03badbc9a1390", "patient_id": "sub-olf11", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf11/anat/sub-olf11_T2w.nii.gz"}, {"path": "sub-olf11/anat/sub-olf11_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf11/anat/sub-olf11_run-01_T1w.nii.gz", "bytes": 29156485, "checksum": "sha256:0d311f59a35fbac3c7a151d9f5439cd1f4b4d10aeeef6a33e516f98fd3ca611b", "patient_id": "sub-olf11", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf11/anat/sub-olf11_run-01_T1w.nii.gz"}, {"path": "sub-olf11/anat/sub-olf11_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf11/anat/sub-olf11_run-02_T1w.nii.gz", "bytes": 32414647, "checksum": "sha256:c938d9415ed53f4288ce57c29094279d5609b16dbd9009bd59fd1d967aa5baf1", "patient_id": "sub-olf11", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf11/anat/sub-olf11_run-02_T1w.nii.gz"}, {"path": "sub-olf12/anat/sub-olf12_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf12/anat/sub-olf12_T2w.nii.gz", "bytes": 58224705, "checksum": "sha256:1dc2a0bc69f84bbd674f250ca383960f1128a9cfc93374dc9d617deb011d6214", "patient_id": "sub-olf12", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf12/anat/sub-olf12_T2w.nii.gz"}, {"path": "sub-olf12/anat/sub-olf12_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf12/anat/sub-olf12_run-01_T1w.nii.gz", "bytes": 28506363, "checksum": "sha256:f279ff8712dce1abe75fbca35a88151cd3f4672d8ca2aa0bac9bf13dd221a71b", "patient_id": "sub-olf12", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf12/anat/sub-olf12_run-01_T1w.nii.gz"}, {"path": "sub-olf12/anat/sub-olf12_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf12/anat/sub-olf12_run-02_T1w.nii.gz", "bytes": 32614169, "checksum": "sha256:adaea676d755f13f43bd8f093b57d6f083529d067cc8e9bbde6786895e021c75", "patient_id": "sub-olf12", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf12/anat/sub-olf12_run-02_T1w.nii.gz"}, {"path": "sub-olf13/anat/sub-olf13_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf13/anat/sub-olf13_T2w.nii.gz", "bytes": 44658524, "checksum": "sha256:b7c0d100ca20c6f1f09525e0368dc8a528709051e393aa57de06a26904e85609", "patient_id": "sub-olf13", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf13/anat/sub-olf13_T2w.nii.gz"}, {"path": "sub-olf13/anat/sub-olf13_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf13/anat/sub-olf13_run-01_T1w.nii.gz", "bytes": 26670719, "checksum": "sha256:dc18c1ea0aca12594200937e659c3052087fe67992a3cba2eccf0157732e6d47", "patient_id": "sub-olf13", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf13/anat/sub-olf13_run-01_T1w.nii.gz"}, {"path": "sub-olf13/anat/sub-olf13_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf13/anat/sub-olf13_run-02_T1w.nii.gz", "bytes": 27754246, "checksum": "sha256:dbd757fdd03b9d2b0f2a5e0c9e42aca6c0e7ed62505f4b376e3d28dd745829d0", "patient_id": "sub-olf13", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf13/anat/sub-olf13_run-02_T1w.nii.gz"}, {"path": "sub-olf14/anat/sub-olf14_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf14/anat/sub-olf14_T2w.nii.gz", "bytes": 64374470, "checksum": "sha256:ad693c88d7b80b7407ea9144521e610817e7bf8d82f83988ef08800a05f280ba", "patient_id": "sub-olf14", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf14/anat/sub-olf14_T2w.nii.gz"}, {"path": "sub-olf14/anat/sub-olf14_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf14/anat/sub-olf14_run-01_T1w.nii.gz", "bytes": 29631084, "checksum": "sha256:5ae6083fb0df3f45c09728025a918b0873362861b655cbc4c28bd560ffc30b63", "patient_id": "sub-olf14", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf14/anat/sub-olf14_run-01_T1w.nii.gz"}, {"path": "sub-olf14/anat/sub-olf14_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf14/anat/sub-olf14_run-02_T1w.nii.gz", "bytes": 33065901, "checksum": "sha256:1527947776fedad553eca8146133a2ac314c32b2e168c6294c8ca76d1da07d2f", "patient_id": "sub-olf14", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf14/anat/sub-olf14_run-02_T1w.nii.gz"}, {"path": "sub-olf15/anat/sub-olf15_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf15/anat/sub-olf15_T2w.nii.gz", "bytes": 65779723, "checksum": "sha256:c16055d4fd3fc4a0e020f4b38032f3a0487a482298c037cfe7f02f813f93b039", "patient_id": "sub-olf15", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf15/anat/sub-olf15_T2w.nii.gz"}, {"path": "sub-olf15/anat/sub-olf15_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf15/anat/sub-olf15_run-01_T1w.nii.gz", "bytes": 31277376, "checksum": "sha256:d0223505605123dc6230d01065bfeae2b509617b6f9670bc7a7d7707a961dc1f", "patient_id": "sub-olf15", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf15/anat/sub-olf15_run-01_T1w.nii.gz"}, {"path": "sub-olf15/anat/sub-olf15_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf15/anat/sub-olf15_run-02_T1w.nii.gz", "bytes": 34894590, "checksum": "sha256:91807ef0730f81d063fb0d8d01cd11824cfd41b62e68056b01b2a54abdbbb6d6", "patient_id": "sub-olf15", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf15/anat/sub-olf15_run-02_T1w.nii.gz"}, {"path": "sub-olf16/anat/sub-olf16_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf16/anat/sub-olf16_T2w.nii.gz", "bytes": 47107039, "checksum": "sha256:0cafb730404c8c4d6ce3cc84c663f305eaf9661c550005aebc5cf410d0435047", "patient_id": "sub-olf16", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf16/anat/sub-olf16_T2w.nii.gz"}, {"path": "sub-olf16/anat/sub-olf16_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf16/anat/sub-olf16_run-01_T1w.nii.gz", "bytes": 23320473, "checksum": "sha256:1bed3c17b6a1a595c87057ff9174098937123470438f2e420e04d9eaa7c49016", "patient_id": "sub-olf16", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf16/anat/sub-olf16_run-01_T1w.nii.gz"}, {"path": "sub-olf16/anat/sub-olf16_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf16/anat/sub-olf16_run-02_T1w.nii.gz", "bytes": 25777505, "checksum": "sha256:2dbccebe4aa2fd95ba516c236897143a5cbb455a787377b1fdb8d1683475c319", "patient_id": "sub-olf16", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf16/anat/sub-olf16_run-02_T1w.nii.gz"}, {"path": "sub-olf17/anat/sub-olf17_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf17/anat/sub-olf17_T2w.nii.gz", "bytes": 74713934, "checksum": "sha256:8456ac5b7c492ecf1e3a96b291d92cd9a763e555a726f8fba8e10c89daebe3c9", "patient_id": "sub-olf17", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf17/anat/sub-olf17_T2w.nii.gz"}, {"path": "sub-olf17/anat/sub-olf17_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf17/anat/sub-olf17_run-01_T1w.nii.gz", "bytes": 38947507, "checksum": "sha256:c38be2e9b04052464f8f1f8d11fd1a87baab5de5398f1cbcb5b9d9452ff47c7e", "patient_id": "sub-olf17", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf17/anat/sub-olf17_run-01_T1w.nii.gz"}, {"path": "sub-olf17/anat/sub-olf17_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf17/anat/sub-olf17_run-02_T1w.nii.gz", "bytes": 43991818, "checksum": "sha256:bbb83d82c17485bac9f0d7b29b6d9be514c8dceda86669be80b7193c4bd5b700", "patient_id": "sub-olf17", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf17/anat/sub-olf17_run-02_T1w.nii.gz"}, {"path": "sub-olf18/anat/sub-olf18_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf18/anat/sub-olf18_T2w.nii.gz", "bytes": 66516117, "checksum": "sha256:d2cb61b094206a217e70c9010969cc160d6c4b97d0a5d980db18ff434047f02e", "patient_id": "sub-olf18", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf18/anat/sub-olf18_T2w.nii.gz"}, {"path": "sub-olf18/anat/sub-olf18_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf18/anat/sub-olf18_run-01_T1w.nii.gz", "bytes": 30228691, "checksum": "sha256:943d1da61bc150e0833ce232dda022532aad2a58274ce2aef89a1c81131bcfb6", "patient_id": "sub-olf18", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf18/anat/sub-olf18_run-01_T1w.nii.gz"}, {"path": "sub-olf18/anat/sub-olf18_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf18/anat/sub-olf18_run-02_T1w.nii.gz", "bytes": 33477204, "checksum": "sha256:034600813f64703beb5f801e01e7eff5508413972ee115721114b0975f832926", "patient_id": "sub-olf18", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf18/anat/sub-olf18_run-02_T1w.nii.gz"}, {"path": "sub-olf19/anat/sub-olf19_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf19/anat/sub-olf19_T2w.nii.gz", "bytes": 68157662, "checksum": "sha256:6ddcfbb415449f1012a27266313bd0117739cb13953d58d98503a0076291ac1b", "patient_id": "sub-olf19", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf19/anat/sub-olf19_T2w.nii.gz"}, {"path": "sub-olf19/anat/sub-olf19_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf19/anat/sub-olf19_run-01_T1w.nii.gz", "bytes": 27970297, "checksum": "sha256:0c8b894d5a0d19e3ee62051b42133bcc9b2f4340a2a9e361b147b2e1804955cd", "patient_id": "sub-olf19", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf19/anat/sub-olf19_run-01_T1w.nii.gz"}, {"path": "sub-olf19/anat/sub-olf19_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf19/anat/sub-olf19_run-02_T1w.nii.gz", "bytes": 36646205, "checksum": "sha256:b286abf2474fd2ea4e5107a9325039288b17fe439d8d236656785d8edeea49a5", "patient_id": "sub-olf19", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf19/anat/sub-olf19_run-02_T1w.nii.gz"}, {"path": "sub-olf20/anat/sub-olf20_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf20/anat/sub-olf20_T2w.nii.gz", "bytes": 74078418, "checksum": "sha256:afd37a4d5ecd338c20fd71d2a7a3a10664326947a53b3363351af7e0a23d3b6e", "patient_id": "sub-olf20", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf20/anat/sub-olf20_T2w.nii.gz"}, {"path": "sub-olf20/anat/sub-olf20_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf20/anat/sub-olf20_run-01_T1w.nii.gz", "bytes": 27205472, "checksum": "sha256:cd954a494aa8d06b1e902ffbef7b240dea5a0de78b80afe34287d3af0ccc4baa", "patient_id": "sub-olf20", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf20/anat/sub-olf20_run-01_T1w.nii.gz"}, {"path": "sub-olf20/anat/sub-olf20_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf20/anat/sub-olf20_run-02_T1w.nii.gz", "bytes": 33004680, "checksum": "sha256:00095280041911bf6df06ef55b85ec6e75eb15ba5ace917c024067c9bcd9ae39", "patient_id": "sub-olf20", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf20/anat/sub-olf20_run-02_T1w.nii.gz"}, {"path": "sub-olf21/anat/sub-olf21_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf21/anat/sub-olf21_T2w.nii.gz", "bytes": 58922872, "checksum": "sha256:3e79b92246089e82939c0bc134be94755592581bce9b835db3c94248fd696fd7", "patient_id": "sub-olf21", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf21/anat/sub-olf21_T2w.nii.gz"}, {"path": "sub-olf21/anat/sub-olf21_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf21/anat/sub-olf21_run-01_T1w.nii.gz", "bytes": 28543534, "checksum": "sha256:7279107041249f0f3b2602b3f63b97a278febb23ab4e62210522c65dff73b314", "patient_id": "sub-olf21", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf21/anat/sub-olf21_run-01_T1w.nii.gz"}, {"path": "sub-olf21/anat/sub-olf21_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf21/anat/sub-olf21_run-02_T1w.nii.gz", "bytes": 38501693, "checksum": "sha256:d2da9f594d2b4b0e593b5c9bac62d39f7fbe2045cdc2766e80dddcbcaa6ee765", "patient_id": "sub-olf21", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf21/anat/sub-olf21_run-02_T1w.nii.gz"}, {"path": "sub-olf22/anat/sub-olf22_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf22/anat/sub-olf22_T2w.nii.gz", "bytes": 59689076, "checksum": "sha256:1920479f0d47b5762b0a07e5ae447ca4b2871c553e3a8940cd51db9e902911a9", "patient_id": "sub-olf22", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf22/anat/sub-olf22_T2w.nii.gz"}, {"path": "sub-olf22/anat/sub-olf22_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf22/anat/sub-olf22_run-01_T1w.nii.gz", "bytes": 29765962, "checksum": "sha256:0ed12169f0a6ce6f09b6c85b1190b1fb3015ceb4c136369fb7e7348d43d4714b", "patient_id": "sub-olf22", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf22/anat/sub-olf22_run-01_T1w.nii.gz"}, {"path": "sub-olf22/anat/sub-olf22_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf22/anat/sub-olf22_run-02_T1w.nii.gz", "bytes": 32434598, "checksum": "sha256:a0857925f45f479d8f5ea84be885f38e66f288f58ee181f86a0c08ab9196e47d", "patient_id": "sub-olf22", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf22/anat/sub-olf22_run-02_T1w.nii.gz"}, {"path": "sub-olf23/anat/sub-olf23_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf23/anat/sub-olf23_T2w.nii.gz", "bytes": 57832122, "checksum": "sha256:a0df84611b1c512d5e1fd117b00745a54763be7fccf3c9afd68dbcee4ae3e078", "patient_id": "sub-olf23", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf23/anat/sub-olf23_T2w.nii.gz"}, {"path": "sub-olf23/anat/sub-olf23_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf23/anat/sub-olf23_run-01_T1w.nii.gz", "bytes": 26992412, "checksum": "sha256:e7b3c1b2e619e09992557942501cfe662b672fecbb454591eb25ec9f1c87f52d", "patient_id": "sub-olf23", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf23/anat/sub-olf23_run-01_T1w.nii.gz"}, {"path": "sub-olf23/anat/sub-olf23_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf23/anat/sub-olf23_run-02_T1w.nii.gz", "bytes": 29205413, "checksum": "sha256:ef00ecf44d287be0af94ca9d841ecdb53e6521a29756d98c63c42c0f31fea06e", "patient_id": "sub-olf23", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf23/anat/sub-olf23_run-02_T1w.nii.gz"}, {"path": "sub-olf24/anat/sub-olf24_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf24/anat/sub-olf24_T2w.nii.gz", "bytes": 49715106, "checksum": "sha256:f208faae7cb67a58bd461c93830f78176a68f5d39d7d77f93fdd47fc60427ffb", "patient_id": "sub-olf24", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf24/anat/sub-olf24_T2w.nii.gz"}, {"path": "sub-olf24/anat/sub-olf24_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf24/anat/sub-olf24_run-01_T1w.nii.gz", "bytes": 27004134, "checksum": "sha256:ea292b390948a5267f9b61e0f09b7022fb4940a4af5e9514bd093716984454a1", "patient_id": "sub-olf24", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf24/anat/sub-olf24_run-01_T1w.nii.gz"}, {"path": "sub-olf24/anat/sub-olf24_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf24/anat/sub-olf24_run-02_T1w.nii.gz", "bytes": 30527119, "checksum": "sha256:77a06e1bf868e88fd4dfbf8b89c519150ee4901f5c6a62b5a1427d936f456829", "patient_id": "sub-olf24", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf24/anat/sub-olf24_run-02_T1w.nii.gz"}, {"path": "sub-olf25/anat/sub-olf25_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf25/anat/sub-olf25_T2w.nii.gz", "bytes": 68499569, "checksum": "sha256:9b989d967e04b1548b7f9c9df2903b9e998327a1661e8cc61e39c2c4c60f9a6d", "patient_id": "sub-olf25", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf25/anat/sub-olf25_T2w.nii.gz"}, {"path": "sub-olf25/anat/sub-olf25_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf25/anat/sub-olf25_run-01_T1w.nii.gz", "bytes": 31948601, "checksum": "sha256:002b9ef91781383aec8c258aeaf49d2383381ce37bd0e615533ae62282c5192d", "patient_id": "sub-olf25", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf25/anat/sub-olf25_run-01_T1w.nii.gz"}, {"path": "sub-olf25/anat/sub-olf25_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf25/anat/sub-olf25_run-02_T1w.nii.gz", "bytes": 34918965, "checksum": "sha256:b0ce10836512a25a88d4eb5d759c81e87d8fed5e6397b1c0bc278ddf9f1c0675", "patient_id": "sub-olf25", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf25/anat/sub-olf25_run-02_T1w.nii.gz"}, {"path": "sub-olf26/anat/sub-olf26_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf26/anat/sub-olf26_T2w.nii.gz", "bytes": 54535231, "checksum": "sha256:a96d10561391d635a98cfd0dadd788fe82703e6d629b5b433eb4641588f355e1", "patient_id": "sub-olf26", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf26/anat/sub-olf26_T2w.nii.gz"}, {"path": "sub-olf26/anat/sub-olf26_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf26/anat/sub-olf26_run-01_T1w.nii.gz", "bytes": 23790075, "checksum": "sha256:07b9d5d36d3a98c55633f91b5dfdc7f19b47f99ec64c3a265f7be7ca521a4fe7", "patient_id": "sub-olf26", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf26/anat/sub-olf26_run-01_T1w.nii.gz"}, {"path": "sub-olf26/anat/sub-olf26_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf26/anat/sub-olf26_run-02_T1w.nii.gz", "bytes": 28313576, "checksum": "sha256:bcf3f46ef4e432d78ea4060c24e8a3ee981e46abe103007e7677146342fcf238", "patient_id": "sub-olf26", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf26/anat/sub-olf26_run-02_T1w.nii.gz"}, {"path": "sub-olf27/anat/sub-olf27_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf27/anat/sub-olf27_T2w.nii.gz", "bytes": 54959472, "checksum": "sha256:3e5819af173c97d7ce39539d5473119b5b57f6a46c5e523c2a427bf27f7c004d", "patient_id": "sub-olf27", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf27/anat/sub-olf27_T2w.nii.gz"}, {"path": "sub-olf27/anat/sub-olf27_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf27/anat/sub-olf27_run-01_T1w.nii.gz", "bytes": 27176812, "checksum": "sha256:7e288525644ec59e615b1c8d255fc54af422805745a9dea3087c5cfddf81fbb5", "patient_id": "sub-olf27", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf27/anat/sub-olf27_run-01_T1w.nii.gz"}, {"path": "sub-olf27/anat/sub-olf27_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf27/anat/sub-olf27_run-02_T1w.nii.gz", "bytes": 30153801, "checksum": "sha256:816da1cd1b45538206dde7587a60af95741df4d72bd34b641052e01e92ab686e", "patient_id": "sub-olf27", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf27/anat/sub-olf27_run-02_T1w.nii.gz"}, {"path": "sub-olf28/anat/sub-olf28_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf28/anat/sub-olf28_T2w.nii.gz", "bytes": 66100370, "checksum": "sha256:50379ca14dd5c547e61d3c44ee768f4ffa0f9e5cf054fcfc7a2bcfb729c5ec4b", "patient_id": "sub-olf28", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf28/anat/sub-olf28_T2w.nii.gz"}, {"path": "sub-olf28/anat/sub-olf28_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf28/anat/sub-olf28_run-01_T1w.nii.gz", "bytes": 29348571, "checksum": "sha256:a21d9a23292421f2dd976dd8ac1ed800b7b3cd261805303a951968c9b1a88975", "patient_id": "sub-olf28", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf28/anat/sub-olf28_run-01_T1w.nii.gz"}, {"path": "sub-olf28/anat/sub-olf28_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf28/anat/sub-olf28_run-02_T1w.nii.gz", "bytes": 31684769, "checksum": "sha256:f794ae449d13d43592d15198f5b8c23844aa98da19cca373aa6898b127f9df24", "patient_id": "sub-olf28", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf28/anat/sub-olf28_run-02_T1w.nii.gz"}, {"path": "sub-olf29/anat/sub-olf29_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf29/anat/sub-olf29_T2w.nii.gz", "bytes": 56461067, "checksum": "sha256:71bba79eaa13af44bce127cb8d7ce754ddb6bae43a8ddd284f54f6115b975483", "patient_id": "sub-olf29", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf29/anat/sub-olf29_T2w.nii.gz"}, {"path": "sub-olf29/anat/sub-olf29_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf29/anat/sub-olf29_run-01_T1w.nii.gz", "bytes": 26471615, "checksum": "sha256:d99dd87678feaeeea4e98544773b8063f3603a78a430430dc6cbf745fedfdf43", "patient_id": "sub-olf29", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf29/anat/sub-olf29_run-01_T1w.nii.gz"}, {"path": "sub-olf29/anat/sub-olf29_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf29/anat/sub-olf29_run-02_T1w.nii.gz", "bytes": 29085874, "checksum": "sha256:67626170bbda522c6944aa0acc5b3f89aebaa541c23f7ea25e791a66487f0284", "patient_id": "sub-olf29", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf29/anat/sub-olf29_run-02_T1w.nii.gz"}, {"path": "sub-olf30/anat/sub-olf30_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf30/anat/sub-olf30_T2w.nii.gz", "bytes": 53280720, "checksum": "sha256:7bbfb0b3c3d6c6044f4926eb0070b4ea61c60a09c01109558c2746c7f9815bbc", "patient_id": "sub-olf30", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf30/anat/sub-olf30_T2w.nii.gz"}, {"path": "sub-olf30/anat/sub-olf30_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf30/anat/sub-olf30_run-01_T1w.nii.gz", "bytes": 28907064, "checksum": "sha256:4a465b4ef57a83295363ddd5f40aac670fdc6f337c6dd64b8d66c5b1eb25e2c5", "patient_id": "sub-olf30", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf30/anat/sub-olf30_run-01_T1w.nii.gz"}, {"path": "sub-olf30/anat/sub-olf30_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf30/anat/sub-olf30_run-02_T1w.nii.gz", "bytes": 32120404, "checksum": "sha256:cd5dde1a0d5c260603b877f87f64d98035366ddc6c55e8be9f7e8bbe508aa7ef", "patient_id": "sub-olf30", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf30/anat/sub-olf30_run-02_T1w.nii.gz"}, {"path": "sub-olf31/anat/sub-olf31_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf31/anat/sub-olf31_T2w.nii.gz", "bytes": 59246149, "checksum": "sha256:b8edbaa13ce983b34e1cc5fb9280477a295cb7e95c53363aa86ee8cbcd9e2f27", "patient_id": "sub-olf31", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf31/anat/sub-olf31_T2w.nii.gz"}, {"path": "sub-olf31/anat/sub-olf31_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf31/anat/sub-olf31_run-01_T1w.nii.gz", "bytes": 28422327, "checksum": "sha256:9f4bb21e28de2d4f3899e4d1f1efe4536b4b1ef8ccc340a4bd69c293946e4fb4", "patient_id": "sub-olf31", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf31/anat/sub-olf31_run-01_T1w.nii.gz"}, {"path": "sub-olf31/anat/sub-olf31_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf31/anat/sub-olf31_run-02_T1w.nii.gz", "bytes": 31064301, "checksum": "sha256:d86a806fe65c7946cf1d62806622644f1023e0063d113bed961ac2553df400fb", "patient_id": "sub-olf31", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf31/anat/sub-olf31_run-02_T1w.nii.gz"}, {"path": "sub-olf32/anat/sub-olf32_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf32/anat/sub-olf32_T2w.nii.gz", "bytes": 53025240, "checksum": "sha256:e1557f216464bfd34ac88bb423917758a46d0e753f33f8d0df925604e0e4c67c", "patient_id": "sub-olf32", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf32/anat/sub-olf32_T2w.nii.gz"}, {"path": "sub-olf32/anat/sub-olf32_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf32/anat/sub-olf32_run-01_T1w.nii.gz", "bytes": 25341713, "checksum": "sha256:5b79d66047a18eefc0354902e1fe7862eb28a388bba6277b13a08d7274dd33b3", "patient_id": "sub-olf32", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf32/anat/sub-olf32_run-01_T1w.nii.gz"}, {"path": "sub-olf32/anat/sub-olf32_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf32/anat/sub-olf32_run-02_T1w.nii.gz", "bytes": 29050221, "checksum": "sha256:dcb7669837f72a47bf4812e287a65e2c1986c7ac40755c09c6d5f4d8d490fd8c", "patient_id": "sub-olf32", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf32/anat/sub-olf32_run-02_T1w.nii.gz"}, {"path": "sub-olf33/anat/sub-olf33_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf33/anat/sub-olf33_T2w.nii.gz", "bytes": 56493574, "checksum": "sha256:bf6f628b4b5f9419eea6c919c80d0681b72379a080a376713537b23ba369e564", "patient_id": "sub-olf33", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf33/anat/sub-olf33_T2w.nii.gz"}, {"path": "sub-olf33/anat/sub-olf33_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf33/anat/sub-olf33_run-01_T1w.nii.gz", "bytes": 27943105, "checksum": "sha256:16f53c6517a34c06b92be6ef705490c76f512b3d08bea201b3516b7a4c7faaaa", "patient_id": "sub-olf33", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf33/anat/sub-olf33_run-01_T1w.nii.gz"}, {"path": "sub-olf33/anat/sub-olf33_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf33/anat/sub-olf33_run-02_T1w.nii.gz", "bytes": 30955360, "checksum": "sha256:02c87f36adb0e7c9d183d79d8970d35c2bc1bd2f546b1feb73de713edc507668", "patient_id": "sub-olf33", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf33/anat/sub-olf33_run-02_T1w.nii.gz"}, {"path": "sub-olf34/anat/sub-olf34_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf34/anat/sub-olf34_T2w.nii.gz", "bytes": 54838757, "checksum": "sha256:10b4bc58e1ed973734e8498c07df17b6eca68d2241d21e245c250de661ec4232", "patient_id": "sub-olf34", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf34/anat/sub-olf34_T2w.nii.gz"}, {"path": "sub-olf34/anat/sub-olf34_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf34/anat/sub-olf34_run-01_T1w.nii.gz", "bytes": 28243868, "checksum": "sha256:8dd6756dba2a59e6462d952fa4f035a7973ec12cdb97192a3f1b0a475ce33a79", "patient_id": "sub-olf34", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf34/anat/sub-olf34_run-01_T1w.nii.gz"}, {"path": "sub-olf34/anat/sub-olf34_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf34/anat/sub-olf34_run-02_T1w.nii.gz", "bytes": 32642732, "checksum": "sha256:ee16f7f9edd911b426982b9d56e00118dd7e706567a03402d9f3364c87e789ec", "patient_id": "sub-olf34", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf34/anat/sub-olf34_run-02_T1w.nii.gz"}, {"path": "sub-olf35/anat/sub-olf35_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf35/anat/sub-olf35_T2w.nii.gz", "bytes": 59070935, "checksum": "sha256:9738376d78d92ca38a78be7656362c499a2d1c22e678da83a5e76d86fcd9e09f", "patient_id": "sub-olf35", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf35/anat/sub-olf35_T2w.nii.gz"}, {"path": "sub-olf35/anat/sub-olf35_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf35/anat/sub-olf35_run-01_T1w.nii.gz", "bytes": 29102922, "checksum": "sha256:84c3b48f9d6d63f77bdd218a845ad53f68f9982b091c397dcf485f589b327e4f", "patient_id": "sub-olf35", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf35/anat/sub-olf35_run-01_T1w.nii.gz"}, {"path": "sub-olf35/anat/sub-olf35_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf35/anat/sub-olf35_run-02_T1w.nii.gz", "bytes": 31274432, "checksum": "sha256:5cb9bf8f4cd5baaa555faf5fa4ae58f688e57d9d9fcebc61c26a839356f0da3d", "patient_id": "sub-olf35", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf35/anat/sub-olf35_run-02_T1w.nii.gz"}, {"path": "sub-olf36/anat/sub-olf36_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf36/anat/sub-olf36_T2w.nii.gz", "bytes": 66836810, "checksum": "sha256:85a253cf671db94ac629a64776762a6e26806dc6ae6153bdfe20000624953ac9", "patient_id": "sub-olf36", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf36/anat/sub-olf36_T2w.nii.gz"}, {"path": "sub-olf36/anat/sub-olf36_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf36/anat/sub-olf36_run-01_T1w.nii.gz", "bytes": 34185108, "checksum": "sha256:bb8d20f7bacee56906835e623f920f73c4986ee07acf2952081965933044d76f", "patient_id": "sub-olf36", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf36/anat/sub-olf36_run-01_T1w.nii.gz"}, {"path": "sub-olf36/anat/sub-olf36_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf36/anat/sub-olf36_run-02_T1w.nii.gz", "bytes": 38918522, "checksum": "sha256:423bf5447160574164d616d48095201911fbe16548da59262948b3a2d12ff8ef", "patient_id": "sub-olf36", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf36/anat/sub-olf36_run-02_T1w.nii.gz"}, {"path": "sub-olf37/anat/sub-olf37_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf37/anat/sub-olf37_T2w.nii.gz", "bytes": 62621345, "checksum": "sha256:561232199258838846d6da7fff3cda3db85e8a44294aa14ef56032c1bb7df69d", "patient_id": "sub-olf37", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf37/anat/sub-olf37_T2w.nii.gz"}, {"path": "sub-olf37/anat/sub-olf37_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf37/anat/sub-olf37_run-01_T1w.nii.gz", "bytes": 33733678, "checksum": "sha256:73fefa7b134385ae2b8f577d82a7d97699b80698b3974952637e0ff641877ad1", "patient_id": "sub-olf37", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf37/anat/sub-olf37_run-01_T1w.nii.gz"}, {"path": "sub-olf37/anat/sub-olf37_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf37/anat/sub-olf37_run-02_T1w.nii.gz", "bytes": 37670736, "checksum": "sha256:881e0fb11a1034c7856f23130c935e488c0a23628edc292fa4ec1551cece8adc", "patient_id": "sub-olf37", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf37/anat/sub-olf37_run-02_T1w.nii.gz"}], "limitations": ["Single center, olfactory groove subtype only; not representative of all meningiomas.", "Only baseline structural images and available tumor/edema masks selected; DWI, fMRI, processed diffusion and all postoperative repeats excluded.", "T1 run numbers do not reliably establish pre/post contrast; retain contrast_unspecified until source metadata review.", "Edema masks are not supplied for every patient; missing is not a healthy label."], "expected_min_masks": 37}, "brain_metastasis": {"id": "brain_metastasis_unresolved", "disease": "Brain Metastasis", "eligible": false, "source": "https://imagingdatasets.ucsf.edu/dataset/1", "license": "Unresolved at dataset level; official benchmark README explicitly prohibits commercial use", "reason": "No verified direct automated MRI download selected. The official UCSF-BMSR v1.3 portal requires a reCAPTCHA challenge on Download; its actual MRI endpoint and publisher checksum were not verified. UCSD-BMETS-Longitudinal publishes an Aspera transfer. No challenge is bypassed and no third-party slice mirror is substituted.", "assets": [], "independent_patients": null, "citation": "Rudie et al. UCSF-BMSR MRI Dataset. Radiology: Artificial Intelligence (2024), e230126. doi:10.1148/ryai.230126", "limitations": ["This is an unresolved verification gap, not evidence that no qualifying brain-metastasis dataset exists.", "UCSF-BMSR advertises 412 patients/560 exams but only 461 public exams; independent patient count in that subset is not established. UCSD alternative has 300 subjects and 163 GB, CC BY 4.0; repeat studies are not independent patients."], "candidate_counts": {"UCSF-BMSR": {"full_cohort_patients": 412, "full_cohort_exams": 560, "currently_advertised_public_exams": 461, "independent_patients_in_public_subset": null}, "UCSD-BMETS-Longitudinal": {"patients": 300, "published_bytes_approx": 163000000000}}}, "alzheimers": {"id": "alzheimers_unresolved", "disease": "Alzheimer\'s Disease", "eligible": false, "source": "https://sites.wustl.edu/oasisbrains/", "license": "Not established for a qualifying directly downloadable clinical MRI cohort", "reason": "No clinical in-vivo 3D MRI cohort with reliable diagnosis, current no-registration access, and verified research permissions established. OASIS official pages return 503; ADNI/AIBL access is controlled. OpenNeuro ds004767 is ex-vivo temporal-lobe tissue, unsuitable as a clinical whole-brain disease class.", "assets": [], "independent_patients": null, "citation": "OASIS alternatives require current access/terms verification before selection.", "limitations": ["Do not label every OASIS subject as Alzheimer disease or equate CDR alone with biomarker-confirmed diagnosis.", "Zenodo 7048297 contains a PDF, not the claimed MRI dataset; public article access does not establish data access.", "No claim that no qualifying dataset exists; selection is unresolved."]}, "post_stroke": {"id": "isles2022", "disease": "Post-stroke Lesions", "eligible": true, "selection": "Verified public multi-center annotated stroke alternative; acute lesions, not chronic T1-only ATLAS", "source": "https://zenodo.org/records/7153326", "metadata_source": "https://zenodo.org/api/records/7153326", "license": "CC-BY-4.0", "license_url": "https://creativecommons.org/licenses/by/4.0/", "citation": "Hernandez Petzsche et al. ISLES 2022: A multi-center magnetic resonance imaging stroke lesion segmentation dataset. Scientific Data 9, 762 (2022). doi:10.1038/s41597-022-01875-5; dataset doi:10.5281/zenodo.7153326", "independent_patients": 250, "expected_groups": 250, "download_bytes": 1692717470, "download_probe": "HEAD 200 with matching 1692717470 byte size. Official open-access Zenodo record and publisher MD5 verified; no payload fetched in Cloud.", "assets": [{"path": "ISLES-2022.zip", "url": "https://zenodo.org/api/records/7153326/files/ISLES-2022.zip/content", "bytes": 1692717470, "checksum": "md5:302ee280373cdd5c190ab763d72a7a50", "archive": true}], "limitations": ["250 public training cases, not the 400-case full challenge cohort.", "Acute/subacute ischemic lesions; not a substitute for chronic post-stroke T1 structural lesions.", "FLAIR, DWI and ADC remain in native space; do not assume cross-sequence registration.", "No normal/control subjects; IXI differs in modality, age distribution and scanner protocol."], "expected_min_masks": 250}}')


In [ ]:
"""Colab-only acquisition helpers. Importing this module never downloads data."""
from __future__ import annotations

import csv
import hashlib
import json
import os
import re
import shutil
import stat
import tarfile
import tempfile
import time
import urllib.error
import urllib.request
import zipfile
from pathlib import Path, PurePosixPath


def digest(path, algorithm="sha256"):
    h = hashlib.new(algorithm)
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(4 * 1024**2), b""):
            h.update(chunk)
    return h.hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    content = json.dumps(value, indent=2) + "\n"
    if path.is_symlink():
        raise ValueError(f"Refusing metadata symlink: {path}")
    if path.exists():
        if path.read_text() == content:
            return
        # Retain prior provenance and failure reports on reruns.
        backup = path.with_name(path.name + f".previous.{time.time_ns()}")
        with path.open("rb") as source, backup.open("xb") as output:
            shutil.copyfileobj(source, output)
    with tempfile.NamedTemporaryFile(mode="w", dir=path.parent, prefix=path.name + ".", suffix=".tmp", delete=False) as stream:
        stream.write(content)
        tmp = Path(stream.name)
    tmp.replace(path)


def checked_path(root, relative):
    if Path(root).is_symlink():
        raise ValueError(f"Refusing destination directory symlink: {root}")
    root = Path(root).resolve()
    p = PurePosixPath(relative)
    if not p.parts or p.is_absolute() or ".." in p.parts or "\\" in relative or ":" in relative:
        raise ValueError(f"Unsafe archive/path member: {relative}")
    target = root.joinpath(*p.parts)
    if not target.resolve().is_relative_to(root):
        raise ValueError(f"Path escapes destination: {relative}")
    return target


def require_space(root, needed, reserve=2 * 1024**3):
    free = shutil.disk_usage(root).free
    if free < needed + reserve:
        raise OSError(f"Insufficient filesystem space: need {needed + reserve:,} bytes; free {free:,}")


def assert_colab_drive(root):
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise RuntimeError("Dataset acquisition is allowed only in Google Colab.") from exc
    mount = Path("/content/drive/MyDrive")
    root = Path(root).resolve()
    if not mount.is_dir() or not root.is_relative_to(mount.resolve()):
        raise RuntimeError("Mount Drive and choose a destination inside /content/drive/MyDrive.")
    # Protect the entire existing IXI tree, not just its Healthy/T2 child.
    protected = Path("/content/drive/MyDrive/BrainMRI/datasets/IXI").resolve()
    if root == protected or root.is_relative_to(protected) or protected.is_relative_to(root):
        raise ValueError("Destination overlaps protected IXI storage.")
    return root


def _request(url, **kwargs):
    if not url.startswith("https://"):
        raise ValueError("Only verified HTTPS sources are allowed.")
    req = urllib.request.Request(url, headers={"User-Agent": "NeuroScope-Acquisition/1.0", "Accept-Encoding": "identity", **kwargs.pop("headers", {})}, **kwargs)
    response = urllib.request.urlopen(req, timeout=90)
    if not response.url.startswith("https://"):
        response.close()
        raise ValueError("Refusing a redirect that downgrades HTTPS.")
    return response


def probe_assets(spec):
    """HEAD only. A failed probe never launches a speculative payload transfer."""
    if not spec.get("eligible"):
        raise RuntimeError(spec["reason"])
    results = []
    for asset in spec["assets"]:
        with _request(asset["url"], method="HEAD") as response:
            if response.status != 200:
                raise RuntimeError(f"HEAD returned {response.status}")
            content_type = response.headers.get("Content-Type", "").lower()
            if any(t in content_type for t in ("text/html", "application/json")):
                raise RuntimeError("HEAD returned a login/error page")
            size = response.headers.get("Content-Length")
            if not size:
                raise RuntimeError("Source did not provide Content-Length; cannot preflight storage")
            size = int(size)
            if asset.get("bytes") is not None and asset["bytes"] != size:
                raise ValueError(f"Source size changed for {asset['path']}")
            asset["bytes"] = size
            results.append({"path": asset["path"], "bytes": size, "accept_ranges": response.headers.get("Accept-Ranges"), "final_url": response.url})
    return results


def download_verified(asset, root, *, max_bytes):
    """Resume only with a server validator and matching Content-Range; never trust ETag as MD5."""
    target = checked_path(root, asset["path"])
    algorithm, expected = asset["checksum"].split(":", 1)
    expected_size = asset.get("bytes")
    length = {"md5": 32, "sha256": 64}.get(algorithm)
    if length is None or not re.fullmatch(r"[0-9a-f]{" + str(length) + r"}", expected):
        raise ValueError("Missing/invalid publisher checksum")
    if target.exists():
        if (expected_size is None or target.stat().st_size == expected_size) and digest(target, algorithm) == expected:
            return {"path": str(target), "sha256": digest(target), "status": "verified_existing"}
        raise FileExistsError(f"Existing file differs from publisher checksum; preserve and inspect: {target}")
    target.parent.mkdir(parents=True, exist_ok=True)
    part = target.with_name(target.name + ".part")
    state_path = target.with_name(target.name + ".resume.json")
    if part.exists() and (expected_size is None or part.stat().st_size == expected_size) and digest(part, algorithm) == expected:
        part.replace(target)
        state_path.unlink(missing_ok=True)
        return {"path": str(target), "sha256": digest(target), "status": "verified_completed_partial"}
    state = json.loads(state_path.read_text()) if state_path.exists() else {}
    headers = {}
    offset = part.stat().st_size if part.exists() else 0
    validator = state.get("validator") if state.get("url") == asset["url"] else None
    if offset and validator:
        headers = {"Range": f"bytes={offset}-", "If-Range": validator}
    elif offset:
        raise RuntimeError(f"Partial file has no trusted resume validator; preserve it and inspect: {part}")
    with _request(asset["url"], headers=headers) as response:
        if response.status not in (200, 206):
            raise RuntimeError(f"Unexpected HTTP status {response.status}")
        if any(x in response.headers.get("Content-Type", "").lower() for x in ["text/html", "application/json"]):
            raise RuntimeError("Source returned a web page/API error instead of the MRI asset")
        if offset and response.status == 206:
            content_range = response.headers.get("Content-Range", "")
            match = re.fullmatch(r"bytes (\d+)-(\d+)/(\d+)", content_range)
            if not match or int(match[1]) != offset:
                raise RuntimeError("Server returned an invalid resume Content-Range")
            total = int(match[3])
            mode = "ab"
        else:
            if response.status == 206:
                raise RuntimeError("Unexpected partial response for a fresh download")
            # If-Range mismatch/server ignores Range: explicitly restart the partial file.
            offset = 0
            mode = "wb"
            total = int(response.headers["Content-Length"]) if response.headers.get("Content-Length") else None
        if expected_size is not None and total is not None and total != expected_size:
            raise ValueError(f"Publisher size differs: expected {expected_size}, server {total}")
        if total is not None and total > max_bytes:
            raise OSError("Asset exceeds the configured download budget")
        limit = expected_size or total or max_bytes
        if limit > max_bytes:
            raise OSError("Asset exceeds the configured download budget")
        require_space(root, max(0, limit - offset))
        etag = response.headers.get("ETag", "")
        validator = etag if etag and not etag.startswith("W/") else response.headers.get("Last-Modified")
        atomic_json(state_path, {"url": asset["url"], "validator": validator, "total": total})
        count = offset
        with part.open(mode) as stream:
            for chunk in iter(lambda: response.read(4 * 1024**2), b""):
                count += len(chunk)
                if count > limit:
                    raise OSError("Download exceeded expected size/budget")
                stream.write(chunk)
        for size in (total, expected_size):
            if size is not None and count != size:
                raise ValueError(f"Incomplete download: {count} != {size}")
    if digest(part, algorithm) != expected:
        raise ValueError(f"Publisher {algorithm} mismatch; file kept as .part for investigation")
    part.replace(target)
    state_path.unlink(missing_ok=True)
    return {"path": str(target), "bytes": count, "publisher_checksum": asset["checksum"], "sha256": digest(target), "status": "verified"}


def safe_extract(archive, destination, *, max_unpacked_bytes):
    """Preflight all members; reject links, devices, traversal, duplicates and archive bombs."""
    archive, destination = Path(archive), Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    if zipfile.is_zipfile(archive):
        container = zipfile.ZipFile(archive)
        members = container.infolist()
        records = []
        for m in members:
            mode = m.external_attr >> 16
            if (stat.S_IFMT(mode) not in (0, stat.S_IFREG, stat.S_IFDIR)) or m.flag_bits & 1:
                container.close()
                raise ValueError("ZIP contains a link, special file or encrypted entry")
            records.append((m.filename, m.file_size, m.is_dir(), m))
        opener = container.open
    else:
        container = tarfile.open(archive, "r:*")
        records = []
        for m in container.getmembers():
            if not (m.isfile() or m.isdir()):
                container.close()
                raise ValueError("TAR contains a link or special file")
            records.append((m.name, m.size, m.isdir(), m))
        opener = container.extractfile
    try:
        total = 0
        seen = set()
        for name, size, is_dir, _ in records:
            if name.rstrip("/") in (".", "") and is_dir:
                continue
            target = checked_path(destination, name)
            key = str(target.resolve()).casefold()
            if key in seen:
                raise ValueError(f"Duplicate archive member: {name}")
            seen.add(key)
            total += size
            if total > max_unpacked_bytes:
                raise OSError("Archive exceeds extraction budget")
        require_space(destination, total)
        for name, size, is_dir, member in records:
            if name.rstrip("/") in (".", "") and is_dir:
                continue
            target = checked_path(destination, name)
            if is_dir:
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            if target.exists():
                # Rerun/resume extraction without blindly overwriting existing data.
                with opener(member) as source:
                    h = hashlib.sha256()
                    for chunk in iter(lambda: source.read(4 * 1024**2), b""):
                        h.update(chunk)
                if target.stat().st_size == size and digest(target) == h.hexdigest():
                    continue
                raise FileExistsError(f"Extraction would replace different existing data: {target}")
            partial = target.with_name(target.name + ".extracting")
            if partial.exists():
                raise FileExistsError(f"Previous interrupted extraction requires inspection: {partial}")
            with opener(member) as source, partial.open("xb") as output:
                shutil.copyfileobj(source, output, length=4 * 1024**2)
            if partial.stat().st_size != size:
                raise ValueError(f"Extracted size mismatch: {name}")
            partial.replace(target)
        return {"members": len(records), "unpacked_bytes": total}
    finally:
        container.close()


def inspect_dataset(root, dataset_id, disease):
    """CPU headers only; one file at a time, no full volume load or GPU preprocessing."""
    import nibabel as nib
    rows, errors = [], []
    for path in sorted(Path(root).rglob("*")):
        if any(p.startswith(".") or p == "__MACOSX" for p in path.relative_to(root).parts):
            continue
        if not path.is_file() or not path.name.endswith((".nii", ".nii.gz")):
            continue
        relative = path.relative_to(root).as_posix()
        try:
            image = nib.load(path)
            if len(image.shape) < 3 or min(image.shape[:3]) < 2:
                raise ValueError(f"Not a 3D MRI/mask: {image.shape}")
            if dataset_id == "msd_task01":
                match = re.search(r"BRATS_\d+", path.name)
                patient = match[0] if match else None
                modality = ["FLAIR", "T1w", "T1ce", "T2w"] if len(image.shape) == 4 else ["mask"]
                role = "mask" if "labelsTr" in path.parts else "image"
            elif dataset_id == "openneuro_ds007345":
                match = re.search(r"sub-olf\d{2}", relative)
                patient = match[0] if match else None  # 'fu' never becomes a new patient
                role = "mask" if "segmentation" in path.parts else "image"
                modality = ["mask" if role == "mask" else ("T2w" if "_T2w" in path.name else "T1w_contrast_unspecified")]
            else:
                match = re.search(r"sub-strokecase\d+", relative)
                patient = match[0] if match else None
                role = "mask" if "_msk" in path.name else "image"
                modality = [next((m for m in ["FLAIR", "ADC", "dwi"] if m.lower() in path.name.lower()), "mask" if role == "mask" else "unknown")]
            if not patient:
                raise ValueError("Cannot establish patient/case identity from the documented layout")
            zooms = [float(v) for v in image.header.get_zooms()[:3]]
            if any(not (0 < z < 100) for z in zooms):
                raise ValueError(f"Invalid voxel spacing: {zooms}")
            rows.append({"dataset_id": dataset_id, "patient_id": f"{dataset_id}:{patient}", "case_id": patient,
                         "disease": disease, "role": role, "modalities": modality, "relative_path": relative,
                         "shape": list(image.shape), "spacing_mm": zooms, "dtype": str(image.get_data_dtype()),
                         "affine": image.affine.tolist(), "bytes": path.stat().st_size,
                         "sha256": digest(path), "diagnosis_source": "dataset-level cohort; review clinical metadata before training",
                         "split_group": f"{dataset_id}:{patient}", "split": "unassigned"})
        except Exception as exc:
            errors.append({"path": relative, "error": f"{type(exc).__name__}: {exc}"})
    if not rows:
        errors.append({"error": "No valid 3D NIfTI files found"})
    groups = {}
    for row in rows:
        g = groups.setdefault(row["patient_id"], {"patient_id": row["patient_id"], "images": [], "masks": [], "split": "unassigned"})
        g["masks" if row["role"] == "mask" else "images"].append(row["relative_path"])
    if any(not g["images"] for g in groups.values()):
        errors.append({"error": "Mask-only patient/case group found"})
    hashes = {}
    for row in rows:
        if row["role"] == "image":
            hashes.setdefault(row["sha256"], set()).add(row["patient_id"])
    duplicates = [sorted(v) for v in hashes.values() if len(v) > 1]
    return {"files": rows, "patients": list(groups.values()), "errors": errors,
            "exact_duplicate_image_groups": duplicates, "patient_count": len(groups),
            "count_note": "MSD IDs are case groups; independent clinical identity is not proven. No cross-release deduplication is claimed."}


def acquire_one(spec, destination, *, download_budget_bytes, extraction_budget_bytes):
    root = assert_colab_drive(destination)
    if not spec.get("eligible"):
        raise RuntimeError(f"{spec['disease']}: acquisition disabled: {spec['reason']}")
    root.mkdir(parents=True, exist_ok=True)
    atomic_json(root / "source_and_license.json", spec)
    report = {"dataset_id": spec["id"], "status": "running", "assets": [], "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
    try:
        known_total = sum(a.get("bytes") or 0 for a in spec["assets"])
        if known_total > download_budget_bytes:
            raise OSError("Selected assets exceed the total download budget")
        require_space(root, known_total)
        used = 0
        for asset in spec["assets"]:
            result = download_verified(asset, root / "downloads", max_bytes=download_budget_bytes - used)
            report["assets"].append(result)
            used += Path(result["path"]).stat().st_size
            if asset.get("archive"):
                report.setdefault("extraction", []).append(safe_extract(result["path"], root / "data", max_unpacked_bytes=extraction_budget_bytes))
        data_root = root / ("data" if any(a.get("archive") for a in spec["assets"]) else "downloads")
        if spec["id"] == "msd_task01":
            descriptions = list(data_root.rglob("dataset.json"))
            if len(descriptions) != 1:
                raise ValueError("Expected one MSD dataset.json to confirm source license and modalities")
            description = json.loads(descriptions[0].read_text())
            license_text = str(description.get("licence", description.get("license", ""))).lower()
            if not ("cc-by-sa" in license_text or "by-sa/4.0" in license_text):
                raise ValueError(f"Unexpected MSD license declaration: {license_text!r}; review before reuse")
        manifest = inspect_dataset(data_root, spec["id"], spec["disease"])
        atomic_json(root / "file_manifest.json", manifest)
        atomic_json(root / "patient_manifest.json", manifest["patients"])
        expected = spec.get("expected_groups")
        if expected and manifest["patient_count"] != expected:
            raise ValueError(f"Expected {expected} patient/case groups; found {manifest['patient_count']}")
        minimum_masks = spec.get("expected_min_masks", 0)
        if sum(row["role"] == "mask" for row in manifest["files"]) < minimum_masks:
            raise ValueError(f"Expected at least {minimum_masks} lesion masks")
        if manifest["errors"] or manifest["exact_duplicate_image_groups"]:
            raise ValueError("Manifest inspection failed; review file_manifest.json")
        report["status"] = "acquired_and_inspected_not_approved_for_training"
        report["patient_or_case_groups"] = manifest["patient_count"]
    except Exception as exc:
        report["status"] = "failed"
        report["error"] = f"{type(exc).__name__}: {exc}"
        raise
    finally:
        atomic_json(root / "acquisition_report.json", report)
    return report


## CUDA readiness for the future Colab workflow

Acquisition, extraction, hashing and header inspection run on CPU and never load complete MRI volumes. The following tiny artificial-array check verifies CUDA, MONAI device compatibility and AMP. It is **not model training, validation, a benchmark of MRI throughput, or a checkpoint operation**. All current/peak memory and timing below describe only this self-test.

For a later separately approved training task: keep CUDA mandatory; use device-aware MONAI transforms, one-volume/lazy loading, small 3D patches and batch size 1, AMP/GradScaler and gradient accumulation where appropriate. Measure peak VRAM and tune patches below the T4 budget. Save model, optimizer, scaler, epoch and random states atomically in private Drive checkpoints and restore all of them on resume. Actual training throughput and clinical validation metrics must be measured then. No such model or loop is included or started here.


In [ ]:
if not IN_COLAB:
    raise RuntimeError("Run acquisition in Google Colab, not Codex Cloud.")
if not torch.cuda.is_available() or not REQUIRE_GPU:
    raise RuntimeError("Select a T4 CUDA runtime. CPU training fallback is forbidden.")
import time
from monai.transforms import NormalizeIntensity
properties = torch.cuda.get_device_properties(0)
VRAM_BUDGET_GIB = min(13.0, properties.total_memory / 2**30 - 2.0)
if VRAM_BUDGET_GIB <= 0:
    raise RuntimeError("Insufficient GPU memory for the future T4 workflow.")
torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
with torch.no_grad():
    tiny = torch.arange(4096, dtype=torch.float32, device="cuda").reshape(1, 16, 16, 16)
    checked = NormalizeIntensity()(tiny)
    if torch.as_tensor(checked).device.type != "cuda" or not torch.isfinite(checked).all().item():
        raise RuntimeError("MONAI CUDA transform self-test failed.")
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        amp_result = torch.ones((16, 16), device="cuda") @ torch.ones((16, 16), device="cuda")
    if not torch.isfinite(amp_result).all().item():
        raise RuntimeError("CUDA AMP arithmetic self-test failed.")
torch.cuda.synchronize()
print({"gpu": properties.name, "vram_gib": round(properties.total_memory / 2**30, 2),
       "future_vram_budget_gib": VRAM_BUDGET_GIB,
       "self_test_seconds": round(time.perf_counter() - started, 4),
       "allocated_mib": round(torch.cuda.memory_allocated() / 2**20, 2),
       "peak_allocated_mib": round(torch.cuda.max_memory_allocated() / 2**20, 2),
       "training_started": False})
del tiny, checked, amp_result
torch.cuda.empty_cache()


## Probe selected download links and reserve storage

HEAD requests read metadata only. An unavailable endpoint, changed size, missing Content-Length, login page, unresolved license/access choice, missing Drive mount or insufficient quota stops the entire preflight before any MRI transfer. Publisher checksums remain the authority even if a host's ETag differs; multipart S3 ETags are not MD5 checksums.


In [ ]:
selected = [CATALOG[key] for key, enabled in SWITCHES.items() if enabled]
preflight_ok = False
if not selected:
    print("No diseases selected. No MRI files read or downloaded; IXI is untouched.")
else:
    if not IN_COLAB or not DRIVE_MOUNTED or not torch.cuda.is_available():
        raise RuntimeError("Colab CUDA runtime and mounted private Drive are required.")
    for spec in selected:
        if not spec["eligible"]:
            raise RuntimeError(f"{spec['disease']}: {spec['reason']}")
    if min(DRIVE_AVAILABLE_GIB, MAX_DOWNLOAD_GIB_PER_DISEASE, MAX_UNPACKED_GIB_PER_ARCHIVE) <= 0:
        raise ValueError("Enter available Drive quota and positive storage budgets before download.")
    acquisition_root = assert_colab_drive(ACQUISITION_ROOT)
    acquisition_root.mkdir(parents=True, exist_ok=True)
    probes = {}
    try:
        for spec in selected:
            probes[spec["id"]] = probe_assets(spec)
        total_download = sum(asset["bytes"] for spec in selected for asset in spec["assets"])
        archive_count = sum(bool(asset.get("archive")) for spec in selected for asset in spec["assets"])
        reserve = total_download + int(archive_count * MAX_UNPACKED_GIB_PER_ARCHIVE * 2**30) + 2 * 2**30
        if reserve > DRIVE_AVAILABLE_GIB * 2**30:
            raise OSError(f"Configured conservative storage reservation is {reserve / 2**30:.2f} GiB; available quota is lower.")
        for spec in selected:
            if sum(asset["bytes"] for asset in spec["assets"]) > MAX_DOWNLOAD_GIB_PER_DISEASE * 2**30:
                raise OSError(f"{spec['id']} exceeds the per-disease download budget.")
        require_space(acquisition_root, reserve - 2 * 2**30)
        preflight_ok = True
        atomic_json(acquisition_root / "preflight_report.json", {"status": "passed", "probes": probes, "reserved_bytes": reserve})
        print(f"Preflight passed: {total_download / 2**30:.2f} GiB download; reserve {reserve / 2**30:.2f} GiB including extraction budgets.")
    except Exception as exc:
        atomic_json(acquisition_root / "preflight_report.json", {"status": "failed", "probes": probes, "error": f"{type(exc).__name__}: {exc}"})
        raise


## Download, verify, extract safely and inspect

This cell executes the selected transfers **in Colab only**. It streams to private Drive `.part` files, verifies the publisher MD5/SHA-256 before promotion, and extracts with full path/type/size preflight. Links, special files, traversal and duplicate members are rejected. Different existing files are preserved and reported. A failed disease is recorded while other independently selected diseases can finish. Rerun after fixing a failure; incomplete extraction files require inspection rather than silent deletion.

Patient/case manifests group all modalities and masks by source patient/case ID. Baseline/follow-up identity is normalized for meningioma. File hashes, 3D shapes, voxel sizes, affine matrices and modality labels are recorded without loading full volumes. Exact cross-patient duplicate images fail inspection. These checks do not detect every near duplicate, prove clinical identity across BraTS releases, confirm every mask's geometry, or approve diagnostic labels for training.


In [ ]:
results = []
if selected and not preflight_ok:
    raise RuntimeError("Rerun and pass preflight before transferring any MRI data.")
for spec in selected:
    try:
        result = acquire_one(spec, str(Path(ACQUISITION_ROOT) / spec["id"]),
                             download_budget_bytes=int(MAX_DOWNLOAD_GIB_PER_DISEASE * 2**30),
                             extraction_budget_bytes=int(MAX_UNPACKED_GIB_PER_ARCHIVE * 2**30))
        results.append(result)
        print(spec["id"], result["status"], result["patient_or_case_groups"])
    except Exception as exc:
        results.append({"dataset_id": spec["id"], "status": "failed", "error": f"{type(exc).__name__}: {exc}"})
        print(spec["id"], "FAILED:", type(exc).__name__, str(exc))
if selected:
    atomic_json(Path(ACQUISITION_ROOT) / "acquisition_summary.json", results)
    failures = [r for r in results if r["status"] == "failed"]
    if failures:
        raise RuntimeError(f"{len(failures)} disease acquisition(s) failed. Review private Drive reports; no training is authorized.")
print("Acquisition stage finished. No models trained. Dataset and label review/approval is still required.")


## Stop before training

Review the private `source_and_license.json`, `acquisition_report.json`, `patient_manifest.json` and `file_manifest.json` for every selected dataset. Confirm clinical diagnosis, mask-to-image geometry, modality availability and count semantics. Compare subject IDs and provenance against any existing cohorts before creating splits; all visits, channels and slices of a patient must share a split. Near-duplicate and cross-release overlap review remains necessary.

**Do not pool IXI T2 controls with heterogeneous disease modalities without addressing scanner, modality, age and site confounding.** Acquisition checks are not clinical validation. No model training or MRI preprocessing/validation experiment is started here. The user must verify and approve the acquired datasets before any later training task.
